# P22 paired workflow — standalone Colab CPU runner

Run all cells on **free CPU**. No Drive mount, GPU, dataset acquisition, or real training. Setup may download Python 3.11 and four notebook/test packages.

The unchanged 17-cell source notebook runs in a separate Python 3.11 kernel; this does not replace Colab's native kernel. All nine code cells must pass. Saved RNA findings are displayed, not recomputed. Relative links refer to the full repository and are not all bundled. Final cell offers an evidence ZIP.

In [ ]:
#@title Verify embedded supporting files
import hashlib, json, os, subprocess, sys, tempfile
from pathlib import Path

PAYLOAD = {'notebooks/P22_paired_workflow.ipynb': '{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "id": "scope",\n   "metadata": {},\n   "source": [\n    "# P22 — paired RNA+ATAC workflow and evidence\\n",\n    "\\n",\n    "**Research outcome: inconclusive. This notebook does not claim a new biological result.**\\n",\n    "\\n",\n    "This is one entry point for the current workflow: software tests, source preflight, saved RNA findings, paired-input requirements, offline HEAD examples, reviewed code, and next steps. It uses the repository\'s existing implementations instead of maintaining a second copy.\\n",\n    "\\n",\n    "## Run safely\\n",\n    "\\n",\n    "Use this notebook inside the accepted P22 checkout with its existing Python 3.11 environment. Select that kernel, then **Restart Kernel and Run All**. No installation cell is included. A downloaded notebook alone needs the accompanying checkout to run.\\n",\n    "\\n",\n    "The first working cell runs the offline parser/capture/preflight tests. Later cells stop if those tests fail. Temporary test fixtures are allowed. No cell downloads datasets, invokes a live transport, starts Docker, performs real model training, changes approvals, or overwrites the old RNA notebook. Existing real-analysis environment flags do not enable those actions here.\\n",\n    "\\n",\n    "Saved RNA findings are displayed, **not recomputed**. Synthetic header examples are **not observations from GEO**. Local Jupyter kernel traffic is not a dataset request.\\n",\n    "\\n",\n    "The complete historical RNA workflow remains in [the original notebook](../P22_down_syndrome_all_in_one.ipynb). Its live-data modes are separate from this offline notebook."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "setup",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "import hashlib\\n",\n    "import json\\n",\n    "import os\\n",\n    "import platform\\n",\n    "import subprocess\\n",\n    "import sys\\n",\n    "from datetime import UTC, datetime\\n",\n    "from pathlib import Path\\n",\n    "from uuid import uuid4\\n",\n    "\\n",\n    "from IPython.display import Code, Markdown, display\\n",\n    "\\n",\n    "assert sys.version_info[:2] == (3, 11), \\"Select the existing P22 Python 3.11 kernel.\\"\\n",\n    "sys.dont_write_bytecode = True\\n",\n    "cwd = Path.cwd().resolve()\\n",\n    "REPO = next(\\n",\n    "    (path for path in (cwd, *cwd.parents)\\n",\n    "     if (path / \\"scripts/launcher_head_capture.py\\").is_file()\\n",\n    "     and (path / \\"pyproject.toml\\").is_file()),\\n",\n    "    None,\\n",\n    ")\\n",\n    "assert REPO is not None, \\"Run this notebook from the accepted P22 checkout.\\"\\n",\n    "OBSERVED_AT = datetime.now(UTC).isoformat()\\n",\n    "print(json.dumps({\\n",\n    "    \\"python\\": platform.python_version(), \\"executable\\": sys.executable,\\n",\n    "    \\"repository\\": str(REPO), \\"observed_at\\": OBSERVED_AT,\\n",\n    "    \\"mode\\": \\"OFFLINE_EVIDENCE_AND_TESTS\\", \\"live_enabled\\": False,\\n",\n    "}, indent=2))"\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "offline-tests",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "# Test first. This list excludes this notebook\'s own kernel test to avoid recursion.\\n",\n    "test_command = [\\n",\n    "    sys.executable, \\"-m\\", \\"pytest\\", \\"-q\\",\\n",\n    "    \\"tests/test_development_head.py\\",\\n",\n    "    \\"tests/test_development_head_capture.py\\",\\n",\n    "    \\"tests/test_launcher_head_capture.py\\",\\n",\n    "]\\n",\n    "test_environment = dict(os.environ, PYTHONPATH=str(REPO / \\"src\\"),\\n",\n    "                        PYTHONDONTWRITEBYTECODE=\\"1\\", PYTEST_DISABLE_PLUGIN_AUTOLOAD=\\"1\\")\\n",\n    "test_run = subprocess.run(\\n",\n    "    test_command, cwd=REPO, env=test_environment,\\n",\n    "    capture_output=True, text=True, timeout=120, check=False,\\n",\n    ")\\n",\n    "print(test_run.stdout)\\n",\n    "if test_run.stderr:\\n",\n    "    print(test_run.stderr)\\n",\n    "OFFLINE_TESTS_PASSED = test_run.returncode == 0\\n",\n    "assert OFFLINE_TESTS_PASSED, \\"Offline tests failed. Stop; do not proceed to later cells.\\""\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "preflight-heading",\n   "metadata": {},\n   "source": [\n    "## 1. Observe the current local inputs\\n",\n    "\\n",\n    "The preflight examines four pinned files, file/path safety, the output destination, and disk headroom. It does not reserve that destination or prove runtime controls. Insufficient disk or changed evidence is a refusal, not a reason to change the limits."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "preflight",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "assert OFFLINE_TESTS_PASSED\\n",\n    "sys.path.insert(0, str(REPO / \\"scripts\\"))\\n",\n    "import launcher_head_capture\\n",\n    "\\n",\n    "assert Path(launcher_head_capture.__file__).resolve() == REPO / \\"scripts/launcher_head_capture.py\\"\\n",\n    "# Do not create this path. Its absence is part of the preflight contract.\\n",\n    "output_candidate = REPO / \\"reports/generated\\" / (\\"notebook-observation-\\" + uuid4().hex)\\n",\n    "preflight_result = launcher_head_capture.preflight(\\n",\n    "    repo=REPO, output_dir=output_candidate, observed_at=OBSERVED_AT,\\n",\n    ")\\n",\n    "print(json.dumps(preflight_result, indent=2))\\n",\n    "assert preflight_result[\\"status\\"] == \\"OFFLINE_PREFLIGHT_OK\\", \\"Preflight refused. Stop here.\\"\\n",\n    "assert preflight_result[\\"live_allowed\\"] is False\\n",\n    "assert preflight_result[\\"runtime_controls\\"] == \\"UNVERIFIED\\"\\n",\n    "assert preflight_result[\\"output_reserved\\"] is False\\n",\n    "assert not output_candidate.exists()"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "source-heading",\n   "metadata": {},\n   "source": [\n    "## 2. The selected source and unchanged resource limits\\n",\n    "\\n",\n    "The candidate URL is a stored proposal, not a verified download link from this run. The object contents, decoded size, full-load memory, and applicable assay classes remain unknown. The limits for a proposed future HEAD do not authorize an object download."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "source-contract",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "contract = json.loads((REPO / \\"configs/development_object_source_contract.json\\").read_text())\\n",\n    "assert contract[\\"status\\"] == \\"SOURCE_UNRESOLVED\\"\\n",\n    "assert contract[\\"object_inspection_authorized\\"] is False\\n",\n    "assert contract[\\"current_object_permissions\\"] == {\\n",\n    "    \\"network_bytes\\": 0, \\"decoded_bytes\\": 0, \\"output_bytes\\": 0,\\n",\n    "    \\"execution_seconds\\": 0, \\"training_allowed\\": False,\\n",\n    "}\\n",\n    "print(json.dumps({\\n",\n    "    \\"status\\": contract[\\"status\\"],\\n",\n    "    \\"candidate_payload_url\\": contract[\\"candidate_payload_url\\"],\\n",\n    "    \\"candidate_url_basis\\": contract[\\"candidate_url_basis\\"],\\n",\n    "    \\"exact_payload_bytes\\": contract[\\"exact_payload_bytes\\"],\\n",\n    "    \\"expected_decoded_bytes\\": contract[\\"expected_decoded_bytes\\"],\\n",\n    "    \\"expected_peak_memory_bytes\\": contract[\\"expected_peak_memory_bytes\\"],\\n",\n    "    \\"current_object_permissions\\": contract[\\"current_object_permissions\\"],\\n",\n    "    \\"reader_proof\\": contract[\\"reader_proof\\"],\\n",\n    "    \\"future_HEAD_proposal_only\\": contract[\\"single_next_action_proposal\\"],\\n",\n    "}, indent=2))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "rna-heading",\n   "metadata": {},\n   "source": [\n    "## 3. Completed RNA evidence — saved results, not a new run\\n",\n    "\\n",\n    "These two sections reproduce the outcome sections from the unchanged, hash-checked research reports. Software test counts are not biological sample sizes. Reproducibility does not turn an inconclusive result into a positive result."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "saved-rna",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "record_pins = {\\n",\n    "    \\"docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md\\":\\n",\n    "        \\"81bb3c313749f4b2751e6259a0c52544d7b10762baddc80597abd36521b96354\\",\\n",\n    "    \\"docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md\\":\\n",\n    "        \\"a8892ffd454898bcdc5594c79e09d55d07d70398be5ce3f8f9b58cde563edd36\\",\\n",\n    "}\\n",\n    "saved_records = {}\\n",\n    "for relative, expected in record_pins.items():\\n",\n    "    with (REPO / relative).open(\\"rb\\") as handle:\\n",\n    "        raw = handle.read(65537)\\n",\n    "    assert len(raw) <= 65536, \\"Saved record exceeds the notebook read limit.\\"\\n",\n    "    assert hashlib.sha256(raw).hexdigest() == expected, \\"Saved RNA evidence changed.\\"\\n",\n    "    saved_records[relative] = raw.decode(\\"utf-8\\")\\n",\n    "    print(relative, expected)\\n",\n    "\\n",\n    "rna_report = saved_records[\\"docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md\\"]\\n",\n    "influence_report = saved_records[\\"docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md\\"]\\n",\n    "assert \\"## Final evidence package\\" in rna_report\\n",\n    "assert \\"## Execution and verification\\" in influence_report\\n",\n    "display(Markdown(rna_report.split(\\"## Final evidence package\\")[0]))\\n",\n    "display(Markdown(influence_report.split(\\"## Execution and verification\\")[0]))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "paired-heading",\n   "metadata": {},\n   "source": [\n    "## 4. Paired-input and training status\\n",\n    "\\n",\n    "The raw-library pilot preserved paired counts for its tested donor/library slice. That is not an accepted development cohort. The two tested library peak lists had zero exact common regions; that result does not prove that every possible processed object is unusable.\\n",\n    "\\n",\n    "External release/QC semantics and compatible measured ATAC counts remain unresolved. Known study provenance is not a complete specimen-independence audit. Synthetic training, final refits, and one-shot scoring already exercised the software; no real external multimodal evaluation followed.\\n",\n    "\\n",\n    "The authoritative criteria below come from the unchanged object contract. Each criterion must apply to the actual selected cohorts. Missing regions cannot become fabricated zero counts."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "scientific-requirements",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "display(Markdown(\\"\\\\n\\".join(\\n",\n    "    \\"- \\" + requirement\\n",\n    "    for requirement in contract[\\"scientific_acceptance_requirements_unchanged\\"]\\n",\n    ")))\\n",\n    "print(\\"Real paired training allowed:\\", contract[\\"current_object_permissions\\"][\\"training_allowed\\"])"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "demo-heading",\n   "metadata": {},\n   "source": [\n    "## 5. Offline examples of the accepted HEAD code\\n",\n    "\\n",\n    "The following header is an invented fixture from `example.invalid`. Its length of 123 bytes has no connection to the real candidate payload. The parser\'s internal `SOURCE_METADATA_OBSERVED` label describes that fixture only.\\n",\n    "\\n",\n    "The capture-core call supplies **no transport**. It must refuse before any request. Notebook success cannot enable a live connection."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "offline-examples",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "from capture_development_head import PINNED_URL, capture_head\\n",\n    "from validate_development_head import parse_head_evidence\\n",\n    "\\n",\n    "synthetic_header = b\\"HTTP/1.1 200 OK\\\\r\\\\nContent-Length: 123\\\\r\\\\n\\\\r\\\\n\\"\\n",\n    "demo_metadata = parse_head_evidence(\\n",\n    "    synthetic_header, url=\\"https://example.invalid/synthetic.rda.gz\\",\\n",\n    "    observed_at=\\"SYNTHETIC_FIXTURE\\",\\n",\n    ")\\n",\n    "assert demo_metadata[\\"declared_payload_bytes\\"] == 123\\n",\n    "assert demo_metadata[\\"publisher_payload_checksum\\"] is None\\n",\n    "print(json.dumps({\\"evidence_kind\\": \\"SYNTHETIC_DEMO_NOT_GEO_OBSERVATION\\",\\n",\n    "                  \\"parser_result\\": demo_metadata}, indent=2))\\n",\n    "\\n",\n    "capture_refusal = capture_head(url=PINNED_URL, observed_at=OBSERVED_AT)\\n",\n    "assert capture_refusal[\\"capture_status\\"] == \\"REFUSED\\"\\n",\n    "assert capture_refusal[\\"reason\\"] == \\"LIVE_TRANSPORT_NOT_ENABLED\\"\\n",\n    "assert capture_refusal[\\"requests_attempted\\"] == 0\\n",\n    "assert capture_refusal[\\"body_bytes\\"] == 0\\n",\n    "print(json.dumps(capture_refusal, indent=2))\\n",\n    "\\n",\n    "bad_label = launcher_head_capture.preflight(\\n",\n    "    repo=REPO, output_dir=output_candidate, observed_at=\\"bad\\\\nlabel\\",\\n",\n    ")\\n",\n    "assert bad_label[\\"reason\\"] == \\"BAD_LABEL\\"\\n",\n    "assert bad_label[\\"source_hashes\\"] == {}\\n",\n    "assert bad_label[\\"disk_free_bytes\\"] is None\\n",\n    "print(\\"Invalid-label refusal:\\", bad_label[\\"reason\\"])"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "code-heading",\n   "metadata": {},\n   "source": [\n    "## 6. Exact implementation used by this notebook\\n",\n    "\\n",\n    "The three current scripts are displayed below. They remain the source of truth; notebook cells import them. Open the collapsed output to inspect the code. The preflight has no live launcher or CLI."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "reviewed-code",\n   "metadata": {\n    "jupyter": {\n     "outputs_hidden": true\n    }\n   },\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "for relative in (\\n",\n    "    \\"scripts/launcher_head_capture.py\\",\\n",\n    "    \\"scripts/capture_development_head.py\\",\\n",\n    "    \\"scripts/validate_development_head.py\\",\\n",\n    "):\\n",\n    "    display(Markdown(\\"### \\" + relative))\\n",\n    "    display(Code(filename=str(REPO / relative), language=\\"python\\"))"\n   ]\n  },\n  {\n   "cell_type": "markdown",\n   "id": "next-steps",\n   "metadata": {},\n   "source": [\n    "## 7. Next steps and stop conditions\\n",\n    "\\n",\n    "| Step | Required evidence | What success permits |\\n",\n    "| --- | --- | --- |\\n",\n    "| E2-M1 | Reviewed native runtime/control design and bounded offline probe allocations | Implement the live binding |\\n",\n    "| E2-M2 / M3 | Real enforcement of memory/no-swap, one total deadline, output limits, safe cleanup, and trusted TLS/input use | Independent controls review |\\n",\n    "| E2-M4 | Accepted controls and the existing exact HEAD scope approval | One bounded metadata request only |\\n",\n    "| E2-R / E3 | Applicable R workspace/assay reader proof, resource contract, and authorized inspection | Assess actual paired inputs |\\n",\n    "| M1–M5 | Accepted release/QC, donor support, measured common regions, provenance, and frozen protocol | Consider the real internal training run |\\n",\n    "| M6–M8 | Accepted development results, frozen model, and accepted external cohort | One-shot external evaluation |\\n",\n    "\\n",\n    "Stop on missing evidence, changed hashes, failed controls, or unavailable resources. A successful HEAD does not prove valid RNA/ATAC assays. Do not weaken scientific gates to obtain a positive result.\\n",\n    "\\n",\n    "### Existing implementation and detailed records\\n",\n    "\\n",\n    "- [Full plan](../tasks/plan.md) and [ordered checklist](../tasks/todo.md)\\n",\n    "- [Paired-data audit and current execution record](../docs/PAIRED_MULTIOME_AUDIT.md)\\n",\n    "- [Dataset roles and experimental question](../docs/PAIRED_DS_MULTIOME_DATASET_OPTIONS.md)\\n",\n    "- [RNA outcome and original executed-notebook links](../docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md)\\n",\n    "- [RNA donor-influence results](../docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md)\\n",\n    "- [Paired training guide](../docs/PAIRED_MULTIOME_TRAINING.md)\\n",\n    "- [Bounded input audit](../scripts/audit_multiome.py), [paired loader](../src/p22/data/multiome.py), [synthetic benchmark](../scripts/train_multiome.py)\\n",\n    "- [Fold training](../src/p22/eval/multiome_runner.py) and [frozen refit/scoring](../src/p22/eval/multiome_final.py)\\n",\n    "\\n",\n    "These links expose existing work without silently running its acquisition or training commands."\n   ]\n  },\n  {\n   "cell_type": "code",\n   "id": "decision",\n   "metadata": {},\n   "execution_count": null,\n   "outputs": [],\n   "source": [\n    "notebook_decision = {\\n",\n    "    \\"mode\\": \\"OFFLINE_EVIDENCE_AND_TESTS\\",\\n",\n    "    \\"offline_tests_passed\\": OFFLINE_TESTS_PASSED,\\n",\n    "    \\"source_status\\": contract[\\"status\\"],\\n",\n    "    \\"scientific_outcome\\": \\"INCONCLUSIVE\\",\\n",\n    "    \\"live_requests_attempted\\": capture_refusal[\\"requests_attempted\\"],\\n",\n    "    \\"training_performed\\": False,\\n",\n    "    \\"live_allowed\\": preflight_result[\\"live_allowed\\"],\\n",\n    "    \\"output_reserved\\": preflight_result[\\"output_reserved\\"],\\n",\n    "    \\"demo_header_is_synthetic\\": True,\\n",\n    "    \\"next_action\\": \\"Review E2-M1 native controls and bounded offline probe allocations.\\",\\n",\n    "}\\n",\n    "assert not output_candidate.exists()\\n",\n    "print(json.dumps(notebook_decision, indent=2))"\n   ]\n  }\n ],\n "metadata": {\n  "kernelspec": {\n   "display_name": "P22 (Python 3.11)",\n   "language": "python",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python",\n   "version": "3.11"\n  },\n  "p22_scope": "offline_evidence_and_tests"\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}\n', 'pyproject.toml': '[project]\nname = "p22"\nversion = "0.1.0"\ndescription = "Modality-routing research scaffold, synthetic-only until explicit approval"\nreadme = "README.md"\nrequires-python = ">=3.11,<3.12"\ndependencies = [\n    "numpy==1.26.4",\n    "pandas==2.2.2",\n    "scikit-learn==1.7.2",\n    "scipy==1.17.1",\n    "torch==2.8.0",\n    "anndata==0.12.6",\n    "h5py==3.16.0",\n    "openpyxl==3.1.5",\n]\n\n[project.optional-dependencies]\ndev = [\n    "pytest>=8.0,<9.0",\n    "ruff>=0.6,<0.16",\n    "jupyterlab==4.6.1",\n    "nbconvert==7.17.1",\n    "nbclient==0.11.0",\n    "ipykernel==7.3.0",\n    "matplotlib==3.10.9",\n]\n# Restores the packages the legacy Tasic scripts under experiments/ were run with.\n# Not installed by default: the pre-approval sequence never executes those scripts.\nlegacy = [\n    "scanpy==1.11.5",\n    "shap==0.48.0",\n    "umap-learn==0.5.12",\n    "seaborn==0.13.2",\n    "statsmodels==0.14.6",\n    "tqdm==4.68.4",\n]\n\n[build-system]\nrequires = ["hatchling"]\nbuild-backend = "hatchling.build"\n\n[tool.hatch.build.targets.wheel]\npackages = ["src/p22"]\n\n[tool.pytest.ini_options]\nminversion = "8.0"\ntestpaths = ["tests"]\naddopts = "-ra -p no:cacheprovider"\nmarkers = [\n    "slow: long-running check, excluded from fast verification",\n    "integration: end-to-end workflow check across modules",\n]\n\n[tool.ruff]\nline-length = 100\ntarget-version = "py311"\nsrc = ["src", "tests", "scripts"]\nextend-exclude = [\n    "notebooks",\n    "colab_notebooks",\n    "experiments",\n    "legacy",\n    "reports/generated",\n]\n\n[tool.ruff.lint]\nselect = ["E", "F", "I", "B", "UP", "SIM"]\n', 'scripts/launcher_head_capture.py': '"""P22 launcher slice: read-only offline preflight observation.\n\nDiagnostic-only. No timing, race, or native-control proof. Output path is not\nreserved; future launcher must revalidate and own race/resource/authorization\nsafeguards. Not a live runner.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nimport shutil\nimport stat\nfrom pathlib import Path\n\nPINNED_FILES: tuple[tuple[str, str, int], ...] = (\n    (\n        "configs/development_object_source_contract.json",\n        "9a13d8beafa5e00d81f8f985649a6a466c6b4fe3ebf008206f840c32316113f2",\n        6429,\n    ),\n    (\n        "docs/PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md",\n        "66b1d8158f5b6c4260085cddb736db2d59de9df610e5dcac5ccfc2c4f63b18b9",\n        14597,\n    ),\n    (\n        "scripts/capture_development_head.py",\n        "335d35c51b27d48704eaad62a695d74e6e91f456e3148b15abe8d3aff846029c",\n        7088,\n    ),\n    (\n        "scripts/validate_development_head.py",\n        "c761489202fdbfaba5153f7a508aefa19028cf87f681bd473382c6c4c713ac9d",\n        4753,\n    ),\n)\nMAX_FILE_SIZE = 65536\nMIN_DISK_FREE = 10737418240\n\n\ndef _bad_label(observed_at: object) -> bool:\n    if not isinstance(observed_at, str):\n        return True\n    if not 1 <= len(observed_at) <= 64:\n        return True\n    return any(ord(ch) < 32 or 127 <= ord(ch) <= 159 for ch in observed_at)\n\n\ndef _safe_path(p: Path) -> bool:\n    if not p.is_absolute():\n        return False\n    return ".." not in p.parts and "\\0" not in str(p)\n\n\ndef _nonsymlink_ancestors(p: Path) -> bool:\n    cur = p\n    while True:\n        try:\n            if cur.is_symlink():\n                return False\n        except OSError:\n            return False\n        parent = cur.parent\n        if parent == cur:\n            return True\n        cur = parent\n\n\ndef _check_pinned_file(repo: Path, rel: str, expected: str, expected_size: int) -> str | None:\n    target = repo / rel\n    if not _nonsymlink_ancestors(target):\n        return "UNSAFE_PATH"\n    try:\n        flags = os.O_RDONLY | os.O_NOFOLLOW | os.O_NONBLOCK\n    except AttributeError:\n        return "INPUT_UNAVAILABLE"\n    try:\n        fd = os.open(target, flags)\n        try:\n            st = os.fstat(fd)\n            if not stat.S_ISREG(st.st_mode):\n                return "UNSAFE_PATH"\n            if st.st_size > MAX_FILE_SIZE:\n                return "FILE_TOO_LARGE"\n            if st.st_size != expected_size:\n                return "PIN_MISMATCH"\n            data = bytearray()\n            while len(data) <= MAX_FILE_SIZE:\n                chunk = os.read(fd, MAX_FILE_SIZE + 1 - len(data))\n                if not chunk:\n                    break\n                data.extend(chunk)\n            if len(data) > MAX_FILE_SIZE:\n                return "FILE_TOO_LARGE"\n            if len(data) != expected_size or hashlib.sha256(data).hexdigest() != expected:\n                return "PIN_MISMATCH"\n            return None\n        finally:\n            os.close(fd)\n    except OSError:\n        return "INPUT_UNAVAILABLE"\n\n\ndef preflight(*, repo: Path, output_dir: Path, observed_at: str) -> dict:\n    result: dict = {\n        "status": "REFUSED",\n        "reason": None,\n        "observed_at": None,\n        "source_hashes": {},\n        "disk_free_bytes": None,\n        "requests_attempted": 0,\n        "body_bytes": 0,\n        "live_allowed": False,\n        "runtime_controls": "UNVERIFIED",\n        "output_reserved": False,\n        "scientific_gate_effect": "NONE",\n    }\n\n    if _bad_label(observed_at):\n        result["reason"] = "BAD_LABEL"\n        return result\n    result["observed_at"] = observed_at\n\n    if not isinstance(repo, Path) or not isinstance(output_dir, Path):\n        result["reason"] = "UNSAFE_PATH"\n        return result\n    if not _safe_path(repo) or not _safe_path(output_dir):\n        result["reason"] = "UNSAFE_PATH"\n        return result\n    try:\n        if not _nonsymlink_ancestors(repo) or not repo.is_dir():\n            result["reason"] = "UNSAFE_PATH"\n            return result\n\n        generated = repo / "reports" / "generated"\n        if not _nonsymlink_ancestors(generated) or not generated.is_dir():\n            result["reason"] = "UNSAFE_PATH"\n            return result\n        if output_dir.parent != generated:\n            result["reason"] = "UNSAFE_PATH"\n            return result\n        if output_dir.exists() or output_dir.is_symlink():\n            result["reason"] = "OUTPUT_EXISTS"\n            return result\n    except OSError:\n        result["reason"] = "UNSAFE_PATH"\n        return result\n\n    for rel, expected, _size in PINNED_FILES:\n        err = _check_pinned_file(repo, rel, expected, _size)\n        if err is not None:\n            result["reason"] = err\n            return result\n        result["source_hashes"][rel] = expected\n\n    try:\n        free = shutil.disk_usage(repo).free\n    except OSError:\n        result["reason"] = "INPUT_UNAVAILABLE"\n        return result\n    if not isinstance(free, int) or isinstance(free, bool):\n        result["reason"] = "DISK_HEADROOM"\n        return result\n    result["disk_free_bytes"] = free\n    if free < MIN_DISK_FREE:\n        result["reason"] = "DISK_HEADROOM"\n        return result\n\n    result["status"] = "OFFLINE_PREFLIGHT_OK"\n    return result\n', 'scripts/capture_development_head.py': '"""Capture one pinned HEAD header block through an explicit trusted transport.\n\nNo default live transport or CLI. The separately authorized launcher must verify\nsource hashes, TLS configuration, isolation, memory/swap/disk/output controls\nand a hard process watchdog. Python signals can be delayed by C execution.\nbody_bytes counts application reads only; TLS/OS buffering is not measured here.\n"""\n\nimport base64\nimport hashlib\nimport signal as _signal\nimport threading\nfrom time import monotonic as _monotonic\n\nfrom validate_development_head import parse_head_evidence\n\nPINNED_URL = (\n    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/"\n    "GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz"\n)\nPINNED_HOST = "ftp.ncbi.nlm.nih.gov"\nDEADLINE_SECONDS = 15.0\nMAX_HEADER_BYTES = 65536\n\n\ndef _valid_label(value):\n    return (\n        isinstance(value, str)\n        and bool(value)\n        and not any(ord(ch) < 32 or 127 <= ord(ch) <= 159 for ch in value)\n    )\n\n\ndef _report(reason, url, observed_at, raw=b"", attempts=0, metadata=None):\n    return {\n        "capture_status": "REFUSED" if reason else "CAPTURED",\n        "reason": reason,\n        "url": url,\n        "observed_at": observed_at,\n        "requests_attempted": attempts,\n        "header_bytes": len(raw),\n        "raw_header_base64": base64.b64encode(raw).decode("ascii"),\n        "header_record_sha256": hashlib.sha256(raw).hexdigest() if raw else None,\n        "metadata": None if reason else metadata,\n        "body_bytes": 0,\n        "scientific_gate_effect": "NONE",\n    }\n\n\ndef capture_head(*, url: str, observed_at: str, connection_factory=None) -> dict:\n    """Return captured metadata or a bounded-category refusal, never authorization.\n\n    The explicit factory constructs an HTTPSConnection-like object without I/O;\n    request() performs connection/TLS/send. A factory that raises before returning\n    owns cleanup of resources it never handed over. The launcher must supply a\n    verified TLS transport without proxy/retry behavior. No arbitrary URL is allowed.\n    """\n    safe_url = url if isinstance(url, str) and url == PINNED_URL else None\n    safe_time = observed_at if _valid_label(observed_at) else None\n    if safe_url is None or safe_time is None:\n        return _report("BAD_TARGET" if safe_url is None else "BAD_LABEL", safe_url, safe_time)\n    if connection_factory is None:\n        return _report("LIVE_TRANSPORT_NOT_ENABLED", safe_url, safe_time)\n\n    deadline = _monotonic() + DEADLINE_SECONDS\n\n    def remaining():\n        seconds = deadline - _monotonic()\n        if seconds <= 0:\n            raise TimeoutError("capture deadline")\n        return seconds\n\n    # Fail before transport if another timer/handler cannot safely be preserved.\n    try:\n        if (\n            threading.current_thread() is not threading.main_thread()\n            or not all(\n                callable(getattr(_signal, name, None))\n                for name in ("getsignal", "signal", "getitimer", "setitimer")\n            )\n            or _signal.getitimer(_signal.ITIMER_REAL) != (0, 0)\n        ):\n            return _report("DEADLINE_UNAVAILABLE", safe_url, safe_time)\n        previous_handler = _signal.getsignal(_signal.SIGALRM)\n        if not callable(previous_handler) and previous_handler not in (\n            _signal.SIG_DFL,\n            _signal.SIG_IGN,\n        ):\n            return _report("DEADLINE_UNAVAILABLE", safe_url, safe_time)\n    except Exception:\n        return _report("DEADLINE_UNAVAILABLE", safe_url, safe_time)\n\n    def alarm(signum, frame):\n        raise TimeoutError("capture deadline")\n\n    conn = None\n    attempts = 0\n    raw = bytearray()\n    reason = None\n    metadata = None\n    cleanup_reason = None\n    try:\n        try:\n            _signal.signal(_signal.SIGALRM, alarm)\n            _signal.setitimer(_signal.ITIMER_REAL, remaining())\n        except TimeoutError:\n            reason = "DEADLINE_EXPIRED"\n        except Exception:\n            reason = "DEADLINE_UNAVAILABLE"\n\n        if reason is None:\n            try:\n                conn = connection_factory(PINNED_HOST, port=443, timeout=remaining())\n                conn.timeout = remaining()\n                attempts = 1\n                conn.request(\n                    "HEAD",\n                    url.removeprefix(f"https://{PINNED_HOST}"),\n                    body=None,\n                    headers={"Accept-Encoding": "identity", "Connection": "close"},\n                )\n                remaining()\n                # ponytail: recv(1) avoids application-level body overread at this\n                # fixed 64 KiB ceiling; optimize only with equivalent byte-bound proof.\n                while len(raw) < MAX_HEADER_BYTES:\n                    conn.sock.settimeout(remaining())\n                    chunk = conn.sock.recv(1)\n                    raw.extend(chunk)\n                    remaining()\n                    if not chunk:\n                        reason = "TRUNCATION"\n                        break\n                    if raw.endswith(b"\\r\\n\\r\\n"):\n                        break\n                else:\n                    reason = "HEADER_CAP"\n\n                if reason is None:\n                    try:\n                        metadata = parse_head_evidence(\n                            bytes(raw), url=safe_url, observed_at=safe_time\n                        )\n                    except ValueError:\n                        reason = "PARSER_REFUSAL"\n                    if reason is None:\n                        encodings = [\n                            line.split(b":", 1)[1].strip(b" \\t").lower()\n                            for line in raw.split(b"\\r\\n")\n                            if line.lower().startswith(b"content-encoding:")\n                        ]\n                        if encodings not in ([], [b"identity"]):\n                            reason = "ENCODING_REFUSAL"\n                        remaining()\n            except TimeoutError:\n                reason = "DEADLINE_EXPIRED"\n            except OSError:\n                reason = "IO_FAILURE"\n    finally:\n        # Restore after partial setup too. Cleanup errors cannot certify success\n        # or overwrite a primary refusal; KeyboardInterrupt/SystemExit propagate.\n        cleanups = []\n        if conn is not None:\n            cleanups.append(conn.close)\n        cleanups.extend(\n            (\n                lambda: _signal.setitimer(_signal.ITIMER_REAL, 0),\n                lambda: _signal.signal(_signal.SIGALRM, previous_handler),\n            )\n        )\n        for cleanup in cleanups:\n            try:\n                cleanup()\n            except TimeoutError:\n                cleanup_reason = cleanup_reason or "DEADLINE_EXPIRED"\n            except Exception:\n                cleanup_reason = cleanup_reason or "IO_FAILURE"\n\n    if reason is None:\n        reason = cleanup_reason\n        if reason is None:\n            try:\n                remaining()\n            except TimeoutError:\n                reason = "DEADLINE_EXPIRED"\n    return _report(reason, safe_url, safe_time, bytes(raw), attempts, metadata)\n', 'scripts/validate_development_head.py': '"""Validate one captured HTTP/1.x header block; never make a network request.\n\nThis is a strict metadata subset, not a general HTTP client. Duplicate observed\nfields and Transfer-Encoding are refused by policy, not by universal HTTP rules.\n"""\n\nimport hashlib\nimport re\n\nMAX_BYTES = 65536\nMAX_CL_DIGITS = 19\nMAX_CL_VALUE = 2**63 - 1\n\n\ndef _validate_label(value: str, name: str) -> None:\n    if not isinstance(value, str) or not value:\n        raise ValueError(f"{name} must be a non-empty string")\n    if any(ord(c) < 32 or 127 <= ord(c) <= 159 for c in value):\n        raise ValueError(f"{name} contains control characters")\n\n\ndef _parse_content_length(value: str) -> int:\n    if not value:\n        raise ValueError("Content-Length is blank")\n    if len(value) > MAX_CL_DIGITS:\n        raise ValueError("Content-Length has excessive digits")\n    if not value.isascii() or not value.isdigit():\n        raise ValueError("Content-Length contains non-ASCII digits or invalid characters")\n    n = int(value)\n    if n < 1:\n        raise ValueError("Content-Length must be positive")\n    if n > MAX_CL_VALUE:\n        raise ValueError("Content-Length exceeds maximum")\n    return n\n\n\ndef parse_head_evidence(raw_headers: bytes, *, url: str, observed_at: str) -> dict:\n    """Return metadata or raise ValueError for an unsupported/invalid capture.\n\n    URL/time are caller labels, not verified provenance. The size cap applies to\n    this already-captured record; it proves no transport, deadline or RAM bound.\n    body_bytes=0 describes this input only. Length/validators/header hash do not\n    establish payload integrity, contents, decoded size or scientific acceptance.\n    """\n    if not isinstance(raw_headers, bytes):\n        raise ValueError("raw_headers must be bytes")\n    if len(raw_headers) > MAX_BYTES:\n        raise ValueError("raw_headers exceeds maximum size")\n    _validate_label(url, "url")\n    _validate_label(observed_at, "observed_at")\n\n    end = raw_headers.find(b"\\r\\n\\r\\n")\n    if end < 0:\n        raise ValueError("missing final CRLF CRLF terminator")\n    if end != len(raw_headers) - 4:\n        raise ValueError("trailing bytes after headers")\n\n    lines = raw_headers[:end].split(b"\\r\\n")\n    status = re.fullmatch(rb"HTTP/1\\.[01] ([0-9]{3}) [\\t\\x20-\\x7e\\x80-\\xff]*", lines[0])\n    if status is None:\n        raise ValueError("malformed status line")\n    http_status = int(status[1])\n    if http_status != 200:\n        raise ValueError(f"non-200 status: {http_status}")\n\n    headers: dict[str, list[str]] = {}\n    for line in lines[1:]:\n        if not line:\n            raise ValueError("empty header line")\n        if line.startswith(b" ") or line.startswith(b"\\t"):\n            raise ValueError("obsolete folded header")\n        if b"\\r" in line or b"\\n" in line:\n            raise ValueError("bare LF or CR in header")\n        if b":" not in line:\n            raise ValueError("malformed header line")\n        name_bytes, value_bytes = line.split(b":", 1)\n        if re.fullmatch(rb"[!#$%&\'*+\\-.^_`|~0-9A-Za-z]+", name_bytes) is None:\n            raise ValueError(f"invalid header name: {name_bytes!r}")\n        name = name_bytes.decode("ascii")\n        value = value_bytes.decode("latin-1").strip(" \\t")\n        if any((ord(c) < 32 and c != "\\t") or ord(c) == 127 for c in value):\n            raise ValueError(f"control character in header value: {name!r}")\n        headers.setdefault(name.lower(), []).append(value)\n\n    if "transfer-encoding" in headers:\n        raise ValueError("Transfer-Encoding is not allowed")\n\n    cl_values = headers.get("content-length")\n    if cl_values is None:\n        raise ValueError("missing Content-Length")\n    if len(cl_values) > 1:\n        raise ValueError("duplicate Content-Length")\n    if "," in cl_values[0]:\n        raise ValueError("Content-Length contains comma list")\n    declared_payload_bytes = _parse_content_length(cl_values[0])\n\n    def _single(name: str) -> str | None:\n        vals = headers.get(name)\n        if vals is None:\n            return None\n        if len(vals) > 1:\n            raise ValueError(f"duplicate {name} header")\n        return vals[0]\n\n    return {\n        "record_kind": "development_head_metadata",\n        "status": "SOURCE_METADATA_OBSERVED",\n        "url": url,\n        "observed_at": observed_at,\n        "http_status": 200,\n        "header_record_sha256": hashlib.sha256(raw_headers).hexdigest(),\n        "header_bytes": len(raw_headers),\n        "declared_payload_bytes": declared_payload_bytes,\n        "etag": _single("etag"),\n        "last_modified": _single("last-modified"),\n        "accept_ranges": _single("accept-ranges"),\n        "publisher_payload_checksum": None,\n        "body_bytes": 0,\n        "scientific_gate_effect": "NONE",\n    }\n', 'tests/test_development_head.py': 'import hashlib\nimport importlib.util\nfrom pathlib import Path\n\nimport pytest\n\nspec = importlib.util.spec_from_file_location(\n    "validate_development_head", Path(__file__).parents[1] / "scripts/validate_development_head.py"\n)\nmod = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(mod)\nparse_head_evidence = mod.parse_head_evidence\n\nURL = "https://example.invalid/object.rda.gz"\nTS = "2024-01-01T00:00:00Z"\n\n\ndef _hdr(status: int = 200, extra: str = "") -> bytes:\n    return f"HTTP/1.1 {status} OK\\r\\n{extra}Content-Length: 100\\r\\n\\r\\n".encode("latin-1")\n\n\ndef test_valid_minimal():\n    raw = _hdr()\n    r = parse_head_evidence(raw, url=URL, observed_at=TS)\n    assert r["record_kind"] == "development_head_metadata"\n    assert r["status"] == "SOURCE_METADATA_OBSERVED"\n    assert r["url"] == URL\n    assert r["observed_at"] == TS\n    assert r["http_status"] == 200\n    assert r["header_record_sha256"] == hashlib.sha256(raw).hexdigest()\n    assert r["header_bytes"] == len(raw)\n    assert r["declared_payload_bytes"] == 100\n    assert r["etag"] is None\n    assert r["last_modified"] is None\n    assert r["accept_ranges"] is None\n    assert r["publisher_payload_checksum"] is None\n    assert r["body_bytes"] == 0\n    assert r["scientific_gate_effect"] == "NONE"\n\n\ndef test_mixed_case_headers():\n    raw = (\n        b\'HTTP/1.1 200 OK\\r\\neTaG:\\t "abc" \\t\\r\\n\'\n        b"LAST-modified: Mon, 01 Jan 2024 00:00:00 GMT\\r\\n"\n        b"accept-RANGES: bytes\\r\\ncONTENT-lENGTH: 42\\r\\n\\r\\n"\n    )\n    r = parse_head_evidence(raw, url=URL, observed_at=TS)\n    assert r["etag"] == \'"abc"\'\n    assert r["last_modified"] == "Mon, 01 Jan 2024 00:00:00 GMT"\n    assert r["accept_ranges"] == "bytes"\n    assert r["declared_payload_bytes"] == 42\n\n\n@pytest.mark.parametrize("status", [301, 302, 304, 404, 500])\ndef test_non_200_rejected(status):\n    raw = f"HTTP/1.1 {status} X\\r\\nContent-Length: 10\\r\\n\\r\\n".encode()\n    with pytest.raises(ValueError, match="non-200"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_missing_content_length():\n    raw = b"HTTP/1.1 200 OK\\r\\n\\r\\n"\n    with pytest.raises(ValueError, match="missing Content-Length"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("second", ["100", "200"])\ndef test_duplicate_content_length(second):\n    raw = _hdr(extra=f"content-LENGTH: {second}\\r\\n")\n    with pytest.raises(ValueError, match="duplicate Content-Length"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("val", ["", "  ", "+5", "-5", "1e5", "1,000", "1,1", "1.5", "0", "000"])\ndef test_invalid_content_length(val):\n    raw = f"HTTP/1.1 200 OK\\r\\nContent-Length: {val}\\r\\n\\r\\n".encode("latin-1")\n    with pytest.raises(ValueError):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_unicode_digit_latin1():\n    raw = b"HTTP/1.1 200 OK\\r\\nContent-Length: \\xb25\\r\\n\\r\\n"\n    with pytest.raises(ValueError, match="non-ASCII"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("digits", [20, 5000])\ndef test_excessive_digits(digits):\n    raw = f"HTTP/1.1 200 OK\\r\\nContent-Length: {\'1\' * digits}\\r\\n\\r\\n".encode()\n    with pytest.raises(ValueError, match="excessive"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_max_s64_accepted():\n    raw = f"HTTP/1.1 200 OK\\r\\nContent-Length: {2**63 - 1}\\r\\n\\r\\n".encode()\n    r = parse_head_evidence(raw, url=URL, observed_at=TS)\n    assert r["declared_payload_bytes"] == 2**63 - 1\n\n\ndef test_over_max_s64_rejected():\n    raw = f"HTTP/1.1 200 OK\\r\\nContent-Length: {2**63}\\r\\n\\r\\n".encode()\n    with pytest.raises(ValueError, match="exceeds maximum"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_boundary_65536():\n    raw = _hdr(extra="X-Pad: " + "a" * (65536 - len(_hdr(extra="X-Pad: \\r\\n"))) + "\\r\\n")\n    assert len(raw) == 65536\n    r = parse_head_evidence(raw, url=URL, observed_at=TS)\n    assert r["header_bytes"] == 65536\n\n\ndef test_boundary_65537_rejected():\n    raw = _hdr(extra="X-Pad: " + "a" * (65537 - len(_hdr(extra="X-Pad: \\r\\n"))) + "\\r\\n")\n    assert len(raw) == 65537\n    with pytest.raises(ValueError, match="exceeds maximum size"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("prefix", [" ", "\\t"])\ndef test_folded_header_rejected(prefix):\n    raw = _hdr(extra=f"{prefix}X-Folded: yes\\r\\n")\n    with pytest.raises(ValueError, match="folded"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_bare_lf_rejected():\n    raw = b"HTTP/1.1 200 OK\\nContent-Length: 10\\n\\n"\n    with pytest.raises(ValueError):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("suffix", [b"AB", b"\\r\\n", _hdr()])\ndef test_appended_bytes_rejected(suffix):\n    raw = _hdr() + suffix\n    with pytest.raises(ValueError, match="trailing bytes"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_truncated_terminator():\n    raw = b"HTTP/1.1 200 OK\\r\\nContent-Length: 10\\r\\n"\n    with pytest.raises(ValueError, match="missing final"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("name", ["ETag", "Last-Modified", "Accept-Ranges"])\ndef test_duplicate_optional_rejected(name):\n    raw = _hdr(extra=f"{name}: a\\r\\n{name.lower()}: a\\r\\n")\n    with pytest.raises(ValueError, match=f"duplicate {name.lower()}"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_transfer_encoding_rejected():\n    raw = b"HTTP/1.1 200 OK\\r\\nTransfer-Encoding: chunked\\r\\nContent-Length: 10\\r\\n\\r\\n"\n    with pytest.raises(ValueError, match="Transfer-Encoding"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("name", ["Bad Name", "Bad(Name", "Bad/Name", "Bad\\xffName", "", "Bad\\t"])\ndef test_invalid_header_name(name):\n    raw = _hdr(extra=f"{name}: x\\r\\n")\n    with pytest.raises(ValueError, match="invalid header name"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("control", ["\\x00", "\\x01", "\\x0b", "\\x0c", "\\x7f"])\ndef test_control_byte_in_value(control):\n    raw = _hdr(extra=f"X-Test: a{control}b\\r\\n")\n    with pytest.raises(ValueError, match="control character"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\ndef test_hash_changes_with_bytes():\n    raw1 = _hdr()\n    raw2 = _hdr(extra="X-Extra: 1\\r\\n")\n    r1 = parse_head_evidence(raw1, url=URL, observed_at=TS)\n    r2 = parse_head_evidence(raw2, url=URL, observed_at=TS)\n    assert r1["header_record_sha256"] != r2["header_record_sha256"]\n    assert r1["publisher_payload_checksum"] is None\n    assert r2["publisher_payload_checksum"] is None\n    assert r1["scientific_gate_effect"] == "NONE"\n    assert r2["scientific_gate_effect"] == "NONE"\n\n\n@pytest.mark.parametrize(\n    "url,ts,match",\n    [\n        ("", TS, "url"),\n        (URL, "", "observed_at"),\n        ("a\\x01b", TS, "control"),\n        (URL, "a\\x01b", "control"),\n        ("a\\x7fb", TS, "control"),\n        (URL, "a\\x80b", "control"),\n        ("a\\x9fb", TS, "control"),\n        (None, TS, "url"),\n        (URL, 123, "observed_at"),\n    ],\n)\ndef test_invalid_labels(url, ts, match):\n    with pytest.raises(ValueError, match=match):\n        parse_head_evidence(_hdr(), url=url, observed_at=ts)\n\n\n@pytest.mark.parametrize("version", [b"HTTP/1.0", b"HTTP/1.1"])\n@pytest.mark.parametrize("reason", [b"", b"OK", b"\\t\\x80\\xff"])\ndef test_valid_status_syntax(version, reason):\n    raw = version + b" 200 " + reason + b"\\r\\nContent-Length: 007\\r\\n\\r\\n"\n    assert parse_head_evidence(raw, url=URL, observed_at=TS)["declared_payload_bytes"] == 7\n\n\n@pytest.mark.parametrize(\n    "status",\n    [\n        b"HTTP/1.2 200 OK",\n        b"HTTP/1.1junk 200 OK",\n        b"HTTP/2.0 200 OK",\n        b"HTTP/1.1 200",\n        b"HTTP/1.1  200 OK",\n        b"HTTP/1.1\\t200 OK",\n        b"HTTP/1.1 20\\xb2 OK",\n        b"HTTP/1.1 200 O\\nK",\n        b"HTTP/1.1 200 O\\rK",\n        b"HTTP/1.1 200 O\\x00K",\n        b"HTTP/1.1 200 O\\x7fK",\n    ],\n)\ndef test_invalid_status_syntax(status):\n    raw = status + b"\\r\\nContent-Length: 1\\r\\n\\r\\n"\n    with pytest.raises(ValueError, match="malformed status line"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("raw", ["headers", bytearray(_hdr()), memoryview(_hdr())])\ndef test_non_bytes_rejected(raw):\n    with pytest.raises(ValueError, match="must be bytes"):\n        parse_head_evidence(raw, url=URL, observed_at=TS)\n\n\n@pytest.mark.parametrize("extra", ["No-Colon\\r\\n", "X-Test: a\\nb\\r\\n", "X-Test: a\\rb\\r\\n"])\ndef test_malformed_field_rejected(extra):\n    with pytest.raises(ValueError, match="malformed header|bare LF or CR"):\n        parse_head_evidence(_hdr(extra=extra), url=URL, observed_at=TS)\n\n\ndef test_latin1_values_and_token_names_preserved():\n    raw = _hdr(extra="ETag: \\t\\xa0\\x80\\xff\\t\\xa0 \\t\\r\\n!#$%&\'*+-.^_`|~: ignored\\r\\n")\n    r = parse_head_evidence(raw, url=" caller label ", observed_at=" caller time ")\n    assert r["etag"] == "\\xa0\\x80\\xff\\t\\xa0"\n    assert r["url"] == " caller label "\n    assert r["observed_at"] == " caller time "\n', 'tests/test_development_head_capture.py': '"""Tests for scripts/capture_development_head.py."""\n\nfrom __future__ import annotations\n\nimport base64\nimport hashlib\nimport importlib.util\nimport json\nimport signal\nimport threading\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport pytest\n\nREPO = Path(__file__).resolve().parents[1]\nSCRIPTS = REPO / "scripts"\nPINNED_URL = (\n    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/"\n    "GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz"\n)\nPINNED_TS = "2024-01-01T00:00:00Z"\nDEADLINE = 15.0\n\n\ndef _load_capture(monkeypatch):\n    monkeypatch.syspath_prepend(str(SCRIPTS))\n    spec = importlib.util.spec_from_file_location(\n        "capture_development_head", SCRIPTS / "capture_development_head.py"\n    )\n    mod = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef _make_fake_signal():\n    state = {"handlers": {}, "timers": {}}\n    for sig in (signal.SIGALRM, signal.SIGINT, signal.SIGTERM):\n        state["handlers"][sig] = (\n            signal.default_int_handler if sig == signal.SIGINT else signal.SIG_DFL\n        )\n    state["timers"][signal.ITIMER_REAL] = (0.0, 0.0)\n    state["timers"][signal.ITIMER_PROF] = (0.0, 0.0)\n    state["timers"][signal.ITIMER_VIRTUAL] = (0.0, 0.0)\n\n    def getsignal(sig):\n        return state["handlers"].get(sig)\n\n    def signal_set(sig, handler):\n        prev = state["handlers"].get(sig)\n        state["handlers"][sig] = handler\n        return prev\n\n    def getitimer(which):\n        return state["timers"].get(which, (0.0, 0.0))\n\n    def setitimer(which, seconds, interval=0.0):\n        prev = state["timers"].get(which, (0.0, 0.0))\n        state["timers"][which] = (seconds, interval)\n        return prev\n\n    ns = SimpleNamespace(\n        SIGALRM=signal.SIGALRM,\n        SIG_DFL=signal.SIG_DFL,\n        SIG_IGN=signal.SIG_IGN,\n        ITIMER_REAL=signal.ITIMER_REAL,\n        getsignal=getsignal,\n        signal=signal_set,\n        getitimer=getitimer,\n        setitimer=setitimer,\n    )\n    return ns, state\n\n\ndef _make_clock(start=0.0):\n    state = {"now": start}\n\n    def tick():\n        return state["now"]\n\n    return tick, state\n\n\nclass TransportHarness:\n    def __init__(\n        self,\n        recv_chunks,\n        *,\n        clock_state,\n        request_side_effect=None,\n        close_side_effect=None,\n        recv_cost=0,\n        factory_cost=0,\n        request_cost=0,\n        fail_at=None,\n        recv_side_effect=None,\n    ):\n        self.data = b"".join(recv_chunks)\n        self.position = 0\n        self.clock = clock_state\n        self.recv_cost = recv_cost\n        self.factory_cost = factory_cost\n        self.request_cost = request_cost\n        self.fail_at = fail_at\n        self.recv_side_effect = recv_side_effect\n        self.request_side_effect = request_side_effect\n        self.close_side_effect = close_side_effect\n        self.requests = []\n        self.factory_calls = []\n        self.read_sizes = []\n        self.timeouts = []\n        self.request_timeout = None\n        self.closed = False\n        self.sock = SimpleNamespace(\n            recv=self._recv,\n            settimeout=self.timeouts.append,\n            timeout=None,\n        )\n        self.timeout = None\n\n    def _recv(self, n):\n        assert n == 1\n        self.read_sizes.append(n)\n        self.clock["now"] += self.recv_cost\n        if self.fail_at is not None and self.position >= self.fail_at:\n            raise OSError("injected read failure")\n        chunk = self.data[self.position : self.position + n]\n        self.position += len(chunk)\n        if self.recv_side_effect:\n            self.recv_side_effect()\n        return chunk\n\n    def request(self, method, path, body=None, headers=None):\n        self.request_timeout = self.timeout\n        self.requests.append((method, path, body, headers))\n        self.clock["now"] += self.request_cost\n        if self.request_side_effect:\n            self.request_side_effect()\n\n    def close(self):\n        self.closed = True\n        if self.close_side_effect:\n            self.close_side_effect()\n\n    def factory(self, host, port=443, timeout=None):\n        self.factory_calls.append((host, port, timeout))\n        self.timeout = timeout\n        self.clock["now"] += self.factory_cost\n        return self\n\n\n@pytest.fixture(autouse=True)\ndef deny_live_network(monkeypatch):\n    import socket\n\n    def _deny(*a, **k):\n        raise AssertionError("live network access attempted")\n\n    monkeypatch.setattr(socket, "create_connection", _deny)\n    monkeypatch.setattr(socket, "getaddrinfo", _deny)\n    monkeypatch.setattr(socket.socket, "connect", _deny)\n    monkeypatch.setattr(socket.socket, "connect_ex", _deny)\n\n\n@pytest.fixture\ndef cap_env(monkeypatch):\n    mod = _load_capture(monkeypatch)\n    fake_sig, sig_state = _make_fake_signal()\n    monkeypatch.setattr(mod, "_signal", fake_sig)\n    clock, clock_state = _make_clock()\n    monkeypatch.setattr(mod, "_monotonic", clock)\n    monkeypatch.setattr(mod, "threading", threading)\n    return SimpleNamespace(\n        mod=mod,\n        sig_state=sig_state,\n        clock_state=clock_state,\n        transport=lambda *args, **kwargs: TransportHarness(\n            *args, clock_state=clock_state, **kwargs\n        ),\n    )\n\n\ndef _valid_headers():\n    return (\n        b"HTTP/1.1 200 OK\\r\\n"\n        b"Content-Length: 12345\\r\\n"\n        b\'ETag: "abc"\\r\\n\'\n        b"Last-Modified: Wed, 21 Oct 2015 07:28:00 GMT\\r\\n"\n        b"Accept-Ranges: bytes\\r\\n"\n        b"\\r\\n"\n    )\n\n\ndef _valid_chunks():\n    hdr = _valid_headers()\n    return [hdr[:32], hdr[32:], b"BODY-SENTINEL"]\n\n\ndef _sized_headers(size):\n    prefix = b"HTTP/1.1 200 OK\\r\\nContent-Length: 1\\r\\nX-Pad: "\n    suffix = b"\\r\\n\\r\\n"\n    raw = prefix + b"A" * (size - len(prefix) - len(suffix)) + suffix\n    assert len(raw) == size\n    return raw\n\n\ndef test_valid_capture(cap_env):\n    from validate_development_head import parse_head_evidence\n\n    raw = _valid_headers()\n    harness = cap_env.transport(_valid_chunks())\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL,\n        observed_at=PINNED_TS,\n        connection_factory=harness.factory,\n    )\n    assert result["capture_status"] == "CAPTURED"\n    assert result["requests_attempted"] == 1\n    assert result["header_bytes"] == len(raw)\n    assert result["raw_header_base64"] == base64.b64encode(raw).decode("ascii")\n    assert result["header_record_sha256"] == hashlib.sha256(raw).hexdigest()\n    assert result["body_bytes"] == 0\n    assert result["scientific_gate_effect"] == "NONE"\n    expected_meta = parse_head_evidence(raw, url=PINNED_URL, observed_at=PINNED_TS)\n    assert result["metadata"] == expected_meta\n    assert harness.requests[0][0] == "HEAD"\n    assert harness.requests[0][1] == PINNED_URL.removeprefix("https://ftp.ncbi.nlm.nih.gov")\n    assert harness.requests[0][2] is None\n    assert harness.requests[0][3] == {"Accept-Encoding": "identity", "Connection": "close"}\n    assert harness.closed is True\n    assert harness.factory_calls == [("ftp.ncbi.nlm.nih.gov", 443, 15.0)]\n    assert len(harness.requests) == 1\n    assert harness.data[harness.position :] == b"BODY-SENTINEL"\n    assert cap_env.sig_state["timers"][signal.ITIMER_REAL] == (0.0, 0.0)\n    assert cap_env.sig_state["handlers"][signal.SIGALRM] == signal.SIG_DFL\n\n\n@pytest.mark.parametrize(\n    "url,ts,reason",\n    [\n        ("http://bad.example", PINNED_TS, "BAD_TARGET"),\n        (PINNED_URL, "", "BAD_LABEL"),\n        (PINNED_URL, "bad\\x00ts", "BAD_LABEL"),\n        (PINNED_URL, "bad\\x80ts", "BAD_LABEL"),\n        (PINNED_URL, {"not": "a timestamp"}, "BAD_LABEL"),\n        (None, PINNED_TS, "BAD_TARGET"),\n    ],\n)\ndef test_preflight_refusals(cap_env, url, ts, reason):\n    harness = cap_env.transport([])\n    result = cap_env.mod.capture_head(url=url, observed_at=ts, connection_factory=harness.factory)\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == reason\n    assert result["requests_attempted"] == 0\n    assert harness.requests == []\n    assert harness.factory_calls == []\n    assert result["url" if reason == "BAD_TARGET" else "observed_at"] is None\n\n\ndef test_none_factory_refusal(cap_env):\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=None\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "LIVE_TRANSPORT_NOT_ENABLED"\n    assert result["requests_attempted"] == 0\n\n\ndef test_exact_65536_success(cap_env):\n    from validate_development_head import parse_head_evidence\n\n    raw = _sized_headers(65536)\n    assert len(raw) == 65536\n    chunks = [raw[:1000], raw[1000:]]\n    harness = cap_env.transport(chunks)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "CAPTURED"\n    assert result["header_bytes"] == 65536\n    assert result["metadata"] == parse_head_evidence(raw, url=PINNED_URL, observed_at=PINNED_TS)\n    assert harness.closed is True\n\n\ndef test_header_cap_65537(cap_env):\n    raw = _sized_headers(65537)\n    assert len(raw) == 65537\n    chunks = [raw[:1000], raw[1000:]]\n    harness = cap_env.transport(chunks)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "HEADER_CAP"\n    assert result["header_bytes"] == 65536\n    assert harness.closed is True\n    assert harness.position == 65536\n    assert len(harness.data[harness.position :]) == 1\n\n\ndef test_many_small_headers_overflow(cap_env):\n    lines = [b"HTTP/1.1 200 OK\\r\\n"]\n    total = len(lines[0])\n    i = 0\n    while total < 65536:\n        line = f"X-Header-{i}: value\\r\\n".encode()\n        lines.append(line)\n        total += len(line)\n        i += 1\n    raw = b"".join(lines) + b"\\r\\n"\n    assert len(raw) > 65536\n    chunks = [raw[:1000], raw[1000:]]\n    harness = cap_env.transport(chunks)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "HEADER_CAP"\n    assert result["header_bytes"] == 65536\n\n\n@pytest.mark.parametrize(\n    "raw,reason",\n    [\n        (b"HTTP/1.1 301 Moved\\r\\nLocation: /new\\r\\n\\r\\n", "PARSER_REFUSAL"),\n        (b"HTTP/1.1 100 Continue\\r\\n\\r\\n", "PARSER_REFUSAL"),\n        (b"HTTP/1.1 200 OK\\r\\n\\r\\n", "PARSER_REFUSAL"),\n        (\n            b"HTTP/1.1 200 OK\\r\\nContent-Length: 1\\r\\nContent-Encoding: gzip\\r\\n\\r\\n",\n            "ENCODING_REFUSAL",\n        ),\n        (\n            b"HTTP/1.1 200 OK\\r\\nContent-Length: 1\\r\\nContent-Encoding: identity\\r\\n"\n            b"Content-Encoding: identity\\r\\n\\r\\n",\n            "ENCODING_REFUSAL",\n        ),\n        (b"HTTP/1.1 200 OK\\r\\nContent-Length: 0\\r\\n\\r\\n", "PARSER_REFUSAL"),\n    ],\n)\ndef test_parser_and_encoding_refusals(cap_env, raw, reason):\n    harness = cap_env.transport([raw, b"NEXT-BLOCK-OR-BODY"])\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == reason\n    assert result["requests_attempted"] == 1\n    assert result["header_bytes"] == len(raw)\n    assert harness.closed is True\n    assert harness.position == len(raw)\n    assert len(harness.requests) == 1\n    assert result["header_record_sha256"] == hashlib.sha256(raw).hexdigest()\n\n\ndef test_eof_truncation(cap_env):\n    raw = b"HTTP/1.1 200 OK\\r\\nContent-Length: 1\\r\\n"\n    harness = cap_env.transport([raw])\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "TRUNCATION"\n    assert result["header_bytes"] == len(raw)\n    assert harness.closed is True\n\n\ndef test_io_failure(cap_env):\n    harness = cap_env.transport(_valid_chunks(), fail_at=20)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "IO_FAILURE"\n    assert harness.closed is True\n    assert base64.b64decode(result["raw_header_base64"]) == _valid_headers()[:20]\n    assert result["header_record_sha256"] == hashlib.sha256(_valid_headers()[:20]).hexdigest()\n\n\ndef test_deadline_expired_slow_trickle(cap_env):\n    raw = _valid_headers()\n    chunks = [raw[:10], raw[10:20], raw[20:]]\n    harness = cap_env.transport(chunks, recv_cost=0.5)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "DEADLINE_EXPIRED"\n    assert harness.closed is True\n    assert harness.position == 30\n    assert base64.b64decode(result["raw_header_base64"]) == raw[:30]\n    assert harness.timeouts[0] == 15.0\n    assert harness.timeouts[-1] == 0.5\n\n\ndef test_late_factory_returns_closes_no_request(cap_env):\n    class LateFactory:\n        def __call__(self, host, port=443, timeout=None):\n            cap_env.clock_state["now"] = 20.0\n            conn = SimpleNamespace(\n                request=lambda *a, **k: None,\n                sock=SimpleNamespace(recv=lambda n: b"", settimeout=lambda t: None, timeout=None),\n                close=lambda: setattr(self, "closed", True),\n                timeout=None,\n            )\n            self.closed = False\n            return conn\n\n    factory = LateFactory()\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "DEADLINE_EXPIRED"\n    assert result["requests_attempted"] == 0\n    assert factory.closed is True\n\n\ndef test_nonmain_thread_refusal(cap_env, monkeypatch):\n    class FakeThread:\n        def is_main(self):\n            return False\n\n    monkeypatch.setattr(cap_env.mod.threading, "current_thread", lambda: FakeThread())\n    harness = cap_env.transport([])\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "DEADLINE_UNAVAILABLE"\n    assert result["requests_attempted"] == 0\n\n\n@pytest.mark.parametrize("timer", [(1.0, 0.0), (0.0, 1.0)])\ndef test_active_timer_refusal(cap_env, timer):\n    cap_env.sig_state["timers"][signal.ITIMER_REAL] = timer\n    harness = cap_env.transport([])\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "DEADLINE_UNAVAILABLE"\n    assert result["requests_attempted"] == 0\n    assert cap_env.sig_state["timers"][signal.ITIMER_REAL] == timer\n    assert harness.factory_calls == []\n\n\ndef test_request_exception(cap_env):\n    def boom():\n        raise OSError("req fail")\n\n    harness = cap_env.transport([], request_side_effect=boom)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "IO_FAILURE"\n    assert result["requests_attempted"] == 1\n    assert harness.closed is True\n\n\ndef test_no_live_requests_on_import(cap_env):\n    assert cap_env.mod.capture_head is not None\n    report = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=None\n    )\n    assert report["capture_status"] == "REFUSED"\n    assert report["reason"] == "LIVE_TRANSPORT_NOT_ENABLED"\n    json.dumps(report)\n\n\ndef test_cleanup_failure_refuses_success(cap_env):\n    def fail_close():\n        raise OSError("close fail")\n\n    harness = cap_env.transport(_valid_chunks(), close_side_effect=fail_close)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "IO_FAILURE"\n    assert result["metadata"] is None\n\n\n@pytest.mark.parametrize("stage", ["request", "last_byte", "parse"])\ndef test_late_result_never_succeeds(cap_env, monkeypatch, stage):\n    raw = _valid_headers()\n    harness = cap_env.transport([raw])\n    if stage == "request":\n        harness.request_cost = 15\n    elif stage == "last_byte":\n\n        def arrive_late():\n            if harness.position == len(raw):\n                cap_env.clock_state["now"] = 15\n\n        harness.recv_side_effect = arrive_late\n    else:\n        parse = cap_env.mod.parse_head_evidence\n\n        def slow_parse(*args, **kwargs):\n            result = parse(*args, **kwargs)\n            cap_env.clock_state["now"] = 15\n            return result\n\n        monkeypatch.setattr(cap_env.mod, "parse_head_evidence", slow_parse)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["reason"] == "DEADLINE_EXPIRED"\n    assert result["metadata"] is None\n    assert result["requests_attempted"] == 1\n    assert base64.b64decode(result["raw_header_base64"]) == (b"" if stage == "request" else raw)\n    assert harness.closed\n    assert cap_env.sig_state["timers"][signal.ITIMER_REAL] == (0, 0)\n    assert cap_env.sig_state["handlers"][signal.SIGALRM] == signal.SIG_DFL\n\n\ndef test_one_start_time_across_stages(cap_env):\n    harness = cap_env.transport(_valid_chunks(), factory_cost=2, request_cost=3)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "CAPTURED"\n    assert harness.factory_calls[0][2] == 15\n    assert harness.request_timeout == 13\n    assert set(harness.timeouts) == {10}\n\n\n@pytest.mark.parametrize("failure", ["missing_api", "noncallable_api", "arming"])\ndef test_deadline_setup_refuses_before_factory(cap_env, monkeypatch, failure):\n    sig = cap_env.mod._signal\n    if failure == "missing_api":\n        monkeypatch.delattr(sig, "setitimer")\n    elif failure == "noncallable_api":\n        monkeypatch.setattr(sig, "setitimer", None)\n    else:\n        original = sig.setitimer\n\n        def broken_arm(which, seconds, interval=0):\n            if seconds:\n                raise OSError("cannot arm")\n            return original(which, seconds, interval)\n\n        monkeypatch.setattr(sig, "setitimer", broken_arm)\n    harness = cap_env.transport([])\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["reason"] == "DEADLINE_UNAVAILABLE"\n    assert harness.factory_calls == []\n    assert cap_env.sig_state["timers"][signal.ITIMER_REAL] == (0, 0)\n    assert cap_env.sig_state["handlers"][signal.SIGALRM] == signal.SIG_DFL\n\n\ndef test_cleanup_does_not_hide_primary_refusal(cap_env):\n    def fail_close():\n        raise OSError("close failed")\n\n    harness = cap_env.transport([b"partial"], close_side_effect=fail_close)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["reason"] == "TRUNCATION"\n    assert result["metadata"] is None\n    assert base64.b64decode(result["raw_header_base64"]) == b"partial"\n    assert cap_env.sig_state["timers"][signal.ITIMER_REAL] == (0, 0)\n    assert cap_env.sig_state["handlers"][signal.SIGALRM] == signal.SIG_DFL\n\n\ndef test_harness_preserves_chunks_and_advances_shared_clock():\n    clock = {"now": 0}\n    harness = TransportHarness([b"AB", b"C"], clock_state=clock, recv_cost=0.5)\n    assert harness.sock.recv(1) == b"A"\n    assert harness.sock.recv(1) == b"B"\n    assert clock["now"] == 1\n    assert harness.data[harness.position :] == b"C"\n\n\ndef test_exact_boundary_fixture_is_valid(monkeypatch):\n    monkeypatch.syspath_prepend(str(SCRIPTS))\n    from validate_development_head import parse_head_evidence\n\n    assert (\n        parse_head_evidence(_sized_headers(65536), url=PINNED_URL, observed_at=PINNED_TS)[\n            "header_bytes"\n        ]\n        == 65536\n    )\n\n\ndef test_restore_actual_previous_handler(cap_env):\n    def previous(*_):\n        pass\n\n    cap_env.sig_state["handlers"][signal.SIGALRM] = previous\n    harness = cap_env.transport(_valid_chunks())\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "CAPTURED"\n    assert cap_env.sig_state["handlers"][signal.SIGALRM] is previous\n\n\n@pytest.mark.parametrize("where", ["timer", "handler"])\ndef test_failed_signal_cleanup_cannot_succeed(cap_env, monkeypatch, where):\n    sig = cap_env.mod._signal\n    if where == "timer":\n        original = sig.setitimer\n\n        def fail_cleanup(which, seconds, interval=0):\n            if seconds == 0:\n                raise OSError("disarm failed")\n            return original(which, seconds, interval)\n\n        monkeypatch.setattr(sig, "setitimer", fail_cleanup)\n    else:\n        original = sig.signal\n\n        def fail_restore(which, handler):\n            if handler == signal.SIG_DFL:\n                raise OSError("restore failed")\n            return original(which, handler)\n\n        monkeypatch.setattr(sig, "signal", fail_restore)\n    harness = cap_env.transport(_valid_chunks())\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["capture_status"] == "REFUSED"\n    assert result["reason"] == "IO_FAILURE"\n    assert result["metadata"] is None\n\n\ndef test_slow_cleanup_cannot_succeed(cap_env):\n    def late_close():\n        cap_env.clock_state["now"] = 15\n\n    harness = cap_env.transport(_valid_chunks(), close_side_effect=late_close)\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["reason"] == "DEADLINE_EXPIRED"\n    assert result["metadata"] is None\n\n\n@pytest.mark.parametrize("where", ["factory", "read"])\ndef test_timeout_categories_and_prefix(cap_env, where):\n    harness = cap_env.transport(_valid_chunks())\n\n    def timeout(*args, **kwargs):\n        raise TimeoutError("test timeout")\n\n    factory = harness.factory\n    if where == "factory":\n        factory = timeout\n        expected = b""\n    else:\n        original = harness.sock.recv\n\n        def read(n):\n            if harness.position == 20:\n                timeout()\n            return original(n)\n\n        harness.sock.recv = read\n        expected = _valid_headers()[:20]\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=factory\n    )\n    assert result["reason"] == "DEADLINE_EXPIRED"\n    assert base64.b64decode(result["raw_header_base64"]) == expected\n\n\ndef test_invalid_labels_never_echo_control_text(cap_env):\n    result = cap_env.mod.capture_head(url="wrong", observed_at="bad\\x00time")\n    assert result["url"] is None\n    assert result["observed_at"] is None\n\n\ndef test_unrestorable_handler_refuses_before_transport(cap_env):\n    cap_env.sig_state["handlers"][signal.SIGALRM] = None\n    harness = cap_env.transport([])\n    result = cap_env.mod.capture_head(\n        url=PINNED_URL, observed_at=PINNED_TS, connection_factory=harness.factory\n    )\n    assert result["reason"] == "DEADLINE_UNAVAILABLE"\n    assert harness.factory_calls == []\n    assert cap_env.sig_state["handlers"][signal.SIGALRM] is None\n\n\n# END OF FILE\n', 'tests/test_launcher_head_capture.py': '# tests/test_launcher_head_capture.py\nimport hashlib\nimport json\nimport os\nimport shutil\nimport socket\nimport subprocess\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport pytest\n\nREPO = Path(__file__).resolve().parents[1]\nPINNED = [\n    "configs/development_object_source_contract.json",\n    "docs/PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md",\n    "scripts/capture_development_head.py",\n    "scripts/validate_development_head.py",\n]\nLABEL = "2026-09-08T00:00:00Z"\nHIGH_FREE = 10737418240\nLOW_FREE = 10737418239\n\n\ndef _deny_socket(*args, **kwargs):\n    raise AssertionError("network denied")\n\n\ndef _deny_subprocess(*args, **kwargs):\n    raise AssertionError("subprocess denied")\n\n\n@pytest.fixture(autouse=True)\ndef deny_runtime(monkeypatch):\n    monkeypatch.setattr(socket, "create_connection", _deny_socket)\n    monkeypatch.setattr(socket, "getaddrinfo", _deny_socket)\n    monkeypatch.setattr(socket, "gethostbyname", _deny_socket)\n    monkeypatch.setattr(socket.socket, "connect", _deny_socket)\n    monkeypatch.setattr(socket.socket, "connect_ex", _deny_socket)\n    monkeypatch.setattr(subprocess, "run", _deny_subprocess)\n    monkeypatch.setattr(subprocess, "Popen", _deny_subprocess)\n    monkeypatch.setattr(subprocess, "call", _deny_subprocess)\n    monkeypatch.setattr(subprocess, "check_call", _deny_subprocess)\n    monkeypatch.setattr(subprocess, "check_output", _deny_subprocess)\n\n\n@pytest.fixture\ndef launcher(monkeypatch):\n    monkeypatch.syspath_prepend(str(REPO / "scripts"))\n    import launcher_head_capture\n\n    return launcher_head_capture\n\n\n@pytest.fixture\ndef workspace(tmp_path):\n    root = tmp_path.resolve() / "repo"\n    root.mkdir()\n    for rel in PINNED:\n        target = root / rel\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(REPO / rel, target)\n    gen = root / "reports" / "generated"\n    gen.mkdir(parents=True)\n    return root\n\n\ndef _run(launcher, repo, output, label=LABEL, free=HIGH_FREE, monkeypatch=None):\n    if monkeypatch:\n        monkeypatch.setattr(launcher.shutil, "disk_usage", lambda _p: SimpleNamespace(free=free))\n    result = launcher.preflight(repo=repo, output_dir=output, observed_at=label)\n    assert set(result) == {\n        "status",\n        "reason",\n        "observed_at",\n        "source_hashes",\n        "disk_free_bytes",\n        "requests_attempted",\n        "body_bytes",\n        "live_allowed",\n        "runtime_controls",\n        "output_reserved",\n        "scientific_gate_effect",\n    }\n    assert len(json.dumps(result)) < 4096\n    return result\n\n\ndef _ok(launcher, repo, output, label=LABEL, free=HIGH_FREE, monkeypatch=None):\n    res = _run(launcher, repo, output, label, free, monkeypatch)\n    assert res["status"] == "OFFLINE_PREFLIGHT_OK"\n    assert res["reason"] is None\n    assert res["observed_at"] == label\n    assert res["requests_attempted"] == 0\n    assert res["body_bytes"] == 0\n    assert res["live_allowed"] is False\n    assert res["runtime_controls"] == "UNVERIFIED"\n    assert res["output_reserved"] is False\n    assert res["scientific_gate_effect"] == "NONE"\n    assert res["disk_free_bytes"] == free\n    assert len(res["source_hashes"]) == 4\n    for rel in PINNED:\n        assert res["source_hashes"][rel] == hashlib.sha256((repo / rel).read_bytes()).hexdigest()\n    return res\n\n\ndef _refuse(launcher, repo, output, reason, label=LABEL, free=HIGH_FREE, monkeypatch=None):\n    res = _run(launcher, repo, output, label, free, monkeypatch)\n    assert res["status"] == "REFUSED"\n    assert res["reason"] == reason\n    assert res["observed_at"] == (None if reason == "BAD_LABEL" else label)\n    assert res["requests_attempted"] == 0\n    assert res["body_bytes"] == 0\n    assert res["live_allowed"] is False\n    assert res["runtime_controls"] == "UNVERIFIED"\n    assert res["output_reserved"] is False\n    assert res["scientific_gate_effect"] == "NONE"\n    return res\n\n\ndef test_ok(launcher, workspace, monkeypatch):\n    _ok(launcher, workspace, workspace / "reports" / "generated" / "new", monkeypatch=monkeypatch)\n\n\n@pytest.mark.parametrize("label", ["a", "x" * 64])\ndef test_valid_label_boundaries(launcher, workspace, monkeypatch, label):\n    _ok(\n        launcher,\n        workspace,\n        workspace / "reports/generated/new",\n        label=label,\n        monkeypatch=monkeypatch,\n    )\n\n\n@pytest.mark.parametrize(\n    "label",\n    [\n        "",\n        "x" * 65,\n        "a\\x00b",\n        "a\\x7fb",\n        "a\\x80b",\n        123,\n        None,\n        b"2026",\n        ["2026"],\n    ],\n)\ndef test_bad_label(launcher, workspace, label, monkeypatch):\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "BAD_LABEL",\n        label,\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_bad_label_no_disk(launcher, workspace, monkeypatch):\n    def boom(_p):\n        raise AssertionError("disk should not be checked")\n\n    monkeypatch.setattr(launcher.shutil, "disk_usage", boom)\n    _refuse(launcher, workspace, workspace / "reports/generated/new", "BAD_LABEL", "a\\x00b")\n\n\ndef test_unsafe_path_relative(launcher, workspace, monkeypatch):\n    _refuse(\n        launcher, workspace, Path("reports/generated/new"), "UNSAFE_PATH", monkeypatch=monkeypatch\n    )\n\n\ndef test_unsafe_path_outside(launcher, workspace, tmp_path, monkeypatch):\n    _refuse(launcher, workspace, tmp_path / "out", "UNSAFE_PATH", monkeypatch=monkeypatch)\n\n\ndef test_unsafe_path_dotdot(launcher, workspace, monkeypatch):\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / ".." / "new",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_output_exists(launcher, workspace, monkeypatch):\n    out = workspace / "reports" / "generated" / "new"\n    out.mkdir()\n    _refuse(launcher, workspace, out, "OUTPUT_EXISTS", monkeypatch=monkeypatch)\n\n\ndef test_output_exists_file(launcher, workspace, monkeypatch):\n    out = workspace / "reports" / "generated" / "new"\n    out.write_text("x")\n    _refuse(launcher, workspace, out, "OUTPUT_EXISTS", monkeypatch=monkeypatch)\n\n\ndef test_missing_input(launcher, workspace, monkeypatch):\n    (workspace / PINNED[0]).unlink()\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "INPUT_UNAVAILABLE",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_symlink_input(launcher, workspace, monkeypatch):\n    target = workspace / PINNED[0]\n    target.unlink()\n    (workspace / "link").write_text("x")\n    os.symlink(workspace / "link", target)\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_file_too_large(launcher, workspace, monkeypatch):\n    target = workspace / PINNED[0]\n    target.write_bytes(b"x" * 65537)\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "FILE_TOO_LARGE",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_pin_mismatch(launcher, workspace, monkeypatch):\n    target = workspace / PINNED[0]\n    target.write_bytes(b"tampered")\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "PIN_MISMATCH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_disk_error(launcher, workspace, monkeypatch):\n    def boom(_p):\n        raise OSError("disk error")\n\n    monkeypatch.setattr(launcher.shutil, "disk_usage", boom)\n    _refuse(launcher, workspace, workspace / "reports/generated/new", "INPUT_UNAVAILABLE")\n\n\ndef test_disk_low(launcher, workspace, monkeypatch):\n    result = _refuse(\n        launcher,\n        workspace,\n        workspace / "reports/generated/new",\n        "DISK_HEADROOM",\n        free=LOW_FREE,\n        monkeypatch=monkeypatch,\n    )\n    assert result["disk_free_bytes"] == LOW_FREE\n\n\ndef test_repo_symlink(launcher, tmp_path, monkeypatch):\n    real = tmp_path.resolve() / "real"\n    real.mkdir()\n    for rel in PINNED:\n        target = real / rel\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(REPO / rel, target)\n    (real / "reports" / "generated").mkdir(parents=True)\n    link = tmp_path / "link"\n    os.symlink(real, link)\n    _refuse(\n        launcher,\n        link,\n        link / "reports" / "generated" / "new",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_generated_symlink(launcher, workspace, monkeypatch):\n    gen = workspace / "reports" / "generated"\n    gen.rmdir()\n    os.symlink(workspace / "reports", gen)\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_no_output_created(launcher, workspace, monkeypatch):\n    out = workspace / "reports" / "generated" / "new"\n    _ok(launcher, workspace, out, monkeypatch=monkeypatch)\n    assert not out.exists()\n\n\ndef test_input_unchanged(launcher, workspace, monkeypatch):\n    before = {rel: (workspace / rel).read_bytes() for rel in PINNED}\n    _ok(launcher, workspace, workspace / "reports" / "generated" / "new", monkeypatch=monkeypatch)\n    after = {rel: (workspace / rel).read_bytes() for rel in PINNED}\n    assert before == after\n\n\n@pytest.mark.parametrize("rel", PINNED)\ndef test_tamper_each_file(launcher, workspace, monkeypatch, rel):\n    target = workspace / rel\n    original = target.read_bytes()\n    target.write_bytes(bytes([original[0] ^ 1]) + original[1:])\n    result = _refuse(\n        launcher,\n        workspace,\n        workspace / "reports/generated/new",\n        "PIN_MISMATCH",\n        monkeypatch=monkeypatch,\n    )\n    assert list(result["source_hashes"]) == PINNED[: PINNED.index(rel)]\n\n\n@pytest.mark.parametrize("kind", ["fifo", "directory"])\ndef test_nonregular_input_refused(launcher, workspace, monkeypatch, kind):\n    target = workspace / PINNED[0]\n    target.unlink()\n    if kind == "fifo":\n        os.mkfifo(target)\n    else:\n        target.mkdir()\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports/generated/new",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_source_parent_symlink(launcher, workspace, monkeypatch):\n    real_dir = workspace / "real_configs"\n    real_dir.mkdir()\n    shutil.copyfile(REPO / PINNED[0], real_dir / "development_object_source_contract.json")\n    (workspace / PINNED[0]).unlink()\n    (workspace / "configs").rmdir()\n    os.symlink(real_dir, workspace / "configs")\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_missing_generated_parent(launcher, workspace, monkeypatch):\n    (workspace / "reports" / "generated").rmdir()\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports" / "generated" / "new",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\ndef test_dangling_output_symlink(launcher, workspace, monkeypatch):\n    out = workspace / "reports" / "generated" / "new"\n    os.symlink(workspace / "nonexistent_target", out)\n    _refuse(launcher, workspace, out, "OUTPUT_EXISTS", monkeypatch=monkeypatch)\n\n\n@pytest.mark.parametrize("flag", ["O_NOFOLLOW", "O_NONBLOCK"])\ndef test_missing_safety_flag_refuses_before_open(launcher, workspace, monkeypatch, flag):\n    def forbidden(*args, **kwargs):\n        raise AssertionError("open without required controls")\n\n    monkeypatch.delattr(os, flag)\n    monkeypatch.setattr(os, "open", forbidden)\n    result = launcher.preflight(\n        repo=workspace, output_dir=workspace / "reports/generated/new", observed_at=LABEL\n    )\n    assert result["status"] == "REFUSED"\n    assert result["reason"] == "INPUT_UNAVAILABLE"\n\n\n@pytest.mark.parametrize("operation", ["open", "fstat", "read", "close"])\ndef test_file_io_failure_refuses_and_closes(launcher, workspace, monkeypatch, operation):\n    real_close = os.close\n    closed = []\n\n    def close(fd):\n        real_close(fd)\n        closed.append(fd)\n        if operation == "close":\n            raise OSError("private close detail")\n\n    def failure(*args, **kwargs):\n        raise OSError("private I/O detail")\n\n    with monkeypatch.context() as patch:\n        patch.setattr(os, "close", close)\n        if operation != "close":\n            patch.setattr(os, operation, failure)\n        result = launcher.preflight(\n            repo=workspace, output_dir=workspace / "reports/generated/new", observed_at=LABEL\n        )\n    assert result["status"] == "REFUSED"\n    assert result["reason"] == "INPUT_UNAVAILABLE"\n    assert len(closed) == (0 if operation == "open" else 1)\n    assert result["source_hashes"] == {}\n\n\n@pytest.mark.parametrize("extra", [1, 70000])\ndef test_growing_file_never_accepts_pinned_prefix(launcher, workspace, monkeypatch, extra):\n    data = (workspace / PINNED[0]).read_bytes() + b"x" * extra\n    position = 0\n    requested = []\n\n    def read(fd, size):\n        nonlocal position\n        requested.append((position, size))\n        chunk = data[position : position + size]\n        position += len(chunk)\n        return chunk\n\n    monkeypatch.setattr(os, "read", read)\n    result = launcher.preflight(\n        repo=workspace, output_dir=workspace / "reports/generated/new", observed_at=LABEL\n    )\n    assert result["status"] == "REFUSED"\n    assert result["reason"] == ("FILE_TOO_LARGE" if extra == 70000 else "PIN_MISMATCH")\n    assert position <= 65537\n    assert all(0 < size <= 65537 - offset for offset, size in requested)\n    assert result["source_hashes"] == {}\n\n\n@pytest.mark.parametrize("free", [True, 10737418240.0, "10737418240", None])\ndef test_disk_observation_requires_integer(launcher, workspace, monkeypatch, free):\n    result = _run(\n        launcher, workspace, workspace / "reports/generated/new", free=free, monkeypatch=monkeypatch\n    )\n    assert result["status"] == "REFUSED"\n    assert result["disk_free_bytes"] is None\n\n\ndef test_bad_label_has_no_filesystem_observation(launcher, workspace, monkeypatch):\n    output = workspace / "reports/generated/new"\n\n    def forbidden(*args, **kwargs):\n        raise AssertionError("bad label touched filesystem")\n\n    with monkeypatch.context() as patch:\n        patch.setattr(os, "open", forbidden)\n        patch.setattr(Path, "stat", forbidden)\n        patch.setattr(Path, "lstat", forbidden)\n        patch.setattr(shutil, "disk_usage", forbidden)\n        result = launcher.preflight(repo=workspace, output_dir=output, observed_at="bad\\n")\n    assert result["reason"] == "BAD_LABEL"\n    assert result["observed_at"] is None\n    assert result["source_hashes"] == {}\n    assert result["disk_free_bytes"] is None\n\n\ndef test_nul_output_path_refused(launcher, workspace, monkeypatch):\n    _refuse(\n        launcher,\n        workspace,\n        workspace / "reports/generated/new\\0",\n        "UNSAFE_PATH",\n        monkeypatch=monkeypatch,\n    )\n\n\n@pytest.mark.parametrize("method", ["is_dir", "exists"])\ndef test_path_observation_error_refused(launcher, workspace, monkeypatch, method):\n    output = workspace / "reports/generated/new"\n\n    def failure(*args, **kwargs):\n        raise OSError("private path detail")\n\n    with monkeypatch.context() as patch:\n        patch.setattr(Path, method, failure)\n        result = launcher.preflight(repo=workspace, output_dir=output, observed_at=LABEL)\n    assert result["reason"] == "UNSAFE_PATH"\n    assert result["status"] == "REFUSED"\n', 'configs/development_object_source_contract.json': '{\n  "schema_version": 1,\n  "planning_revision": "2026-09-14",\n  "status": "SOURCE_UNRESOLVED",\n  "phase": "E2_OFFLINE_CONTRACT",\n  "object_inspection_authorized": false,\n  "selected_target": "GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz",\n  "intended_scope": "Lattke excitatory-lineage PCW10-20 development donors; actual coverage unverified",\n  "declared_listing_size": "7.6G (rounded, not an exact byte measurement)",\n  "format_declared": "gzip-compressed R workspace; actual serialized classes unknown",\n  "candidate_payload_url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz",\n  "candidate_url_basis": "Derived from the pinned source note\'s directory and exact filename; not a new HTTP observation",\n  "exact_payload_url": null,\n  "exact_payload_bytes": null,\n  "publisher_payload_checksum": null,\n  "expected_decoded_bytes": null,\n  "expected_peak_memory_bytes": null,\n  "source_evidence": {\n    "path": "docs/PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md",\n    "sha256": "66b1d8158f5b6c4260085cddb736db2d59de9df610e5dcac5ccfc2c4f63b18b9"\n  },\n  "reader_proof": {\n    "image_id": "sha256:264df259ebb2fa4ff2a3db6d346ec273e18d4c84af727b1cc4b39065a9e8edbb",\n    "r_version": "4.6.1",\n    "matrix_version": "1.7.6",\n    "tested_serialization": "saveRDS/readRDS of a 392-byte .rds fixture; not save/load of a .rda workspace",\n    "tested_classes": ["list", "dgCMatrix", "data.frame"],\n    "fixture_report_sha256": "1cf56325eb4f08c86d11661b59ec9985d9300e18a1f9a738c60fed57c9b99813",\n    "not_proven": [".rda workspace loading", "Seurat", "ChromatinAssay", "real-object memory fit", "real raw ATAC assay"]\n  },\n  "current_object_permissions": {\n    "network_bytes": 0,\n    "decoded_bytes": 0,\n    "output_bytes": 0,\n    "execution_seconds": 0,\n    "training_allowed": false\n  },\n  "single_next_action_proposal": {\n    "status": "REQUIRES_SEPARATE_SCOPE_APPROVAL",\n    "purpose": "Observe candidate payload status and available size/validator headers without reading the object body; supersedes the unexecuted listing GET",\n    "method": "HEAD",\n    "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz",\n    "maximum_requests": 1,\n    "redirects": 0,\n    "retries": 0,\n    "maximum_response_header_bytes": 65536,\n    "maximum_response_body_bytes": 0,\n    "maximum_decoded_bytes": 0,\n    "maximum_retained_output_bytes": 1048576,\n    "maximum_process_tree_memory_bytes": 268435456,\n    "maximum_swap_bytes": 0,\n    "maximum_wall_seconds": 15,\n    "minimum_host_free_disk_bytes": 10737418240,\n    "controls": "Proposed pinned cached Python in an inspected non-root container; offline-tested aggregate header cap and absolute deadline; HEAD only with identity HTTP encoding and no body reads, redirects or Range fallback; no host-data mounts; refuse before request if controls are unproven",\n    "preservation": "New output directory only; preserve prior evidence and downloaded R archive; no cache/user-data deletion",\n    "stop_conditions": ["non-200 or redirect", "deadline or header limit", "missing/invalid/non-positive Content-Length", "source evidence changed", "required controls unavailable"],\n    "optional_observations": ["ETag", "Last-Modified", "Accept-Ranges", "publisher checksum if explicitly supplied; never infer one from validators"],\n    "evidence_semantics": "Save URL/time/status/headers and a locally computed header-record SHA-256. Content-Length, if valid, is server-declared GET representation length, not measured payload bytes, decoded size or memory. Optional missing headers remain unknown; validators/header hashes are not payload checksums; Accept-Ranges does not guarantee a future partial response.",\n    "after_success": "E2-R offline target-reader/resource planning only. Source observations do not authorize package installation, fixture execution, GET/Range, object loading or training.",\n    "after_failure": "Record SOURCE_UNRESOLVED and the exact missing field/control or request failure, with a local header-record hash if headers were received; no retry or source substitution"\n  },\n  "target_reader_resource_gate": {\n    "status": "NOT_RUN",\n    "offline_proposal_minutes": 30,\n    "network_install_object_bytes_authorized": 0,\n    "maximum_proposed_fixture_bytes": 1048576,\n    "required_reader_proof": "Applicable save/load .rda workspace and Seurat/ChromatinAssay extraction, or demonstrated equivalent narrow reader; exact sparse counts, IDs and metadata; unexpected classes refuse. Cost missing package assets before acquisition; generic E1 proof is preserved but insufficient.",\n    "required_resource_proposal": "Separate numeric transfer, decoded, retained/temporary disk, process-tree memory/swap and wall caps, current host headroom, evidence-backed full-load estimates and enforceable stop controls. Preserve ledgers and specify class-fixture/dependency allocation or amendment; do not infer memory from compressed length or reset setup budgets.",\n    "stop_rule": "No defensible resource contract yields RESOURCE_UNRESOLVED; missing class support stays explicitly unproven. Installation or runtime setting changes require their own reviewed exact authorization before execution. A real load additionally requires applicable fixture proof and its reviewed exact acquisition/inspection contract.",\n    "forbidden_shortcuts": ["64 KiB prefix/Range probe as assay proof", "compressed bytes below RAM cap as memory-fit proof", "OOM as proof of missing assay", "automatic alternate object or retry"]\n  },\n  "scientific_acceptance_requirements_unchanged": [\n    "paired raw RNA and measured ATAC counts with exact retained barcode/donor join",\n    "release and QC/count-stage semantics",\n    "age/class support and specimen provenance",\n    "genome build, interval coordinates and count units",\n    "exact common measured regions; never zero-fill unmatched peaks",\n    "feature-selection provenance including discovery donors/labels; cohort-wide cluster peaks are not automatically label-independent or train-fold-only, even when disclosed",\n    "post-QC donor/class support for five repeated five-fold internal evaluation and inner validation; two donors per class are insufficient and 13+13 is external margin context, not a development minimum",\n    "separate development, external and cross-cohort decisions"\n  ]\n}\n', 'docs/PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md': '# Remaining paired-multiome evidence: bounded source check\n\nChecked 2026-09-08; public transport/release rechecked 2026-09-09. Public metadata\nand methods only; no model fitting, count-matrix download, controlled access,\nauthor contact, approval change, or exclusion applied.\n\n## Result\n\nTwo gaps narrowed substantially; neither is a completed external-validation run.\n\n| Gate | New evidence | Current status |\n|---|---|---|\n| NeMO release count | Exactly 3,731 metadata rows have RNA annotation `Unk`; the other rows total 113,801 | Exact candidate reconciliation, not verified author exclusion or complete QC replay |\n| Specimen provenance | Author methods name UCLA cores and NIH NeuroBioBank; metadata gives 18 UCLA and 8 NIH donors | Documented provider-level separation from Lattke/HDBR; no overlap evidence found, specimen-level non-overlap not independently certified |\n| Common measured ATAC regions | Both authors describe study-specific peak calling; Lattke source explicitly recounts fragments | No cross-cohort common-count release established; confirmatory compatibility remains pending |\n\n## 1. NeMO count difference has a precise candidate explanation\n\nThe [author manuscript](https://pmc.ncbi.nlm.nih.gov/articles/PMC13225313/) reports\n113,801 retained nuclei. The already-pinned public metadata contains 117,532 rows.\nIts RNA annotations give the following exact partition:\n\n| Metadata selection | Cells | Control cells | Ts21 cells | Control donors | Ts21 donors |\n|---|---:|---:|---:|---:|---:|\n| All rows | 117,532 | 61,656 | 55,876 | 13 | 13 |\n| `class == "Unk"` | 3,731 | 2,107 | 1,624 | 13 | 13 |\n| `class != "Unk"` | 113,801 | 59,549 | 54,252 | 13 | 13 |\n\n`class == "Unk"` and `cluster.ids == "Unk"` identify exactly the same rows.\nEvery donor remains after this candidate filter; the smallest donor retains 273\ncells. This is a deterministic annotation-based reconciliation, not deletion of\nan arbitrary number of cells. It is **not yet proof that this was the authors\'\nfinal exclusion rule**: the retrieved methods do not explicitly identify `Unk`\nas the reason for the reported count.\n\nDo not substitute the WNN annotation: `cluster.ids.wnn == "Unk"` identifies only\n728 rows. Of those, 663 overlap RNA `Unk` and 65 do not. `class.wnn` has no `Unk`\ncategory. RNA and WNN inclusion masks therefore are not interchangeable.\n\n### Additional QC caveat\n\nMethods describe RNA counts above 200 with a donor-relative upper cutoff, ATAC\nfragment counts above 100, and tissue mitochondrial percentage below 5.\n[Source: quality-control methods](https://pmc.ncbi.nlm.nih.gov/articles/PMC13225313/).\n\nThe current metadata does not directly reproduce those thresholds:\n\n| Current metadata | `nCount_ATAC <= 100` | `percent.mt >= 5` | Either condition |\n|---|---:|---:|---:|\n| All rows | 1,798 | 7 | 1,805 |\n| RNA non-`Unk` rows | 1,751 | 6 | 1,757 |\n\nNo row has `nCount_RNA <= 200`. These diagnostics do not establish that the paper\nis wrong: the current ATAC column may refer to a different counting stage or\nregion set, and the release may differ. Those are possible explanations, not\nverified facts. Do not silently reinterpret this column as the original QC\nfragment total, re-estimate the original donor cutoff from a filtered release,\nor claim the 113,801 candidate rows all pass reproduced QC.\n\n**Acceptance still needed:** an author-defined final barcode list, an explicit\nsource-code exclusion rule, or equivalent archive documentation connecting the\nrelease to the final RNA/ATAC analysis. Until then, report\n`ANNOTATION_COUNT_MATCH_QC_UNVERIFIED`; do not promote this diagnostic to M1/M8\nscientific acceptance.\n\n## 2. Tissue providers are now documented\n\nVuong\'s sample-acquisition methods name the UCLA Gene and Cell Therapy Core,\nUCLA Translational Pathology Core Laboratory, and NIH NeuroBioBank repository.\n[Source: sample-acquisition methods](https://pmc.ncbi.nlm.nih.gov/articles/PMC13225313/).\n\nThe downloaded metadata\'s `source` field is invariant within each donor:\n\n| Source | Donors | Control donors | Ts21 donors | All cells | RNA non-`Unk` cells |\n|---|---:|---:|---:|---:|---:|\n| `UCLA` | 18 | 8 | 10 | 74,346 | 71,387 |\n| `NIH_NBB` | 8 | 5 | 3 | 43,186 | 42,414 |\n\n`condition`, `source`, `gw`, `sex`, and `ancestry` each have one value per donor.\nThe NIH donors are `NIH1031`, `NIH1089`, `NIH1139`, `NIH1152`, `NIH1329`,\n`NIH1644`, `NIH1883`, and `NIH908`. The other 18 donor identifiers are UCLA\nrecords. No public crosswalk to HDBR specimens was found in these sources.\n\nLattke\'s fresh-frozen fetal multiome tissue came from the UK Human Developmental\nBiology Resource, project 200585. Its separate Zagreb fixed-tissue material is\nnot the multiome donor source. [Source: fetal-tissue methods](https://www.nature.com/articles/s41591-026-04211-1).\n\n**Conclusion:** `no_overlap_evidence`, supported by different documented\nprocurement providers and the previously checked distinct donor IDs. This is\nstronger than the former donor-name-only check. It is not a genetic identity\ncheck or an author-certified specimen crosswalk; the NIH source field does not\nidentify each original collection site. An unqualified claim of independently\nverified specimen non-overlap would still overstate this evidence.\n\n## 3. Existing common-count objects do not yet solve the feature contract\n\nVuong methods describe GRCh38 2024-A-2.0.0 alignment, merged-fragment MACS2 peak\ncalling, blacklist removal, and a resulting nucleus-by-peaks matrix.\n[Source: preprocessing and integration methods](https://pmc.ncbi.nlm.nih.gov/articles/PMC13225313/).\nThat supports a study-level shared matrix, not identical measured intervals in\nLattke or a fixed reference established independently of all development donors.\n\nThe [pinned Lattke peak-quantification script](https://github.com/lattkem1/Down_Syndrome_Multiome/blob/227f51b4e63c6a7d9c73be44f06ab21ac11e45ba/F_Chromatin_scMEGA_GRN_analysis_exc_lin_from_all_non_cx_excl_v045_scripts/F01_v045_seur_call_quant_peaks_by_cluster.R)\nexplicitly calls peaks by `cluster_name`, then uses\n`FeatureMatrix(fragments = Fragments(seur), features = peaks, cells = colnames(seur))`\nand creates a `peaks_by_cluster` assay. Its output is\n`F01_seur_w_peaks_by_cluster_quant.rda` in an author-local directory. The checked\n[public source tree](https://api.github.com/repos/lattkem1/Down_Syndrome_Multiome/git/trees/227f51b4e63c6a7d9c73be44f06ab21ac11e45ba?recursive=1)\nis complete (`truncated: false`) but does not contain that count checkpoint.\n\nThe [GEO supplementary listing](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305146/suppl/)\ndoes list two processed objects:\n\n- `GSE305146_seur_integr_labelled_complete_dataset.rda.gz` (listing: 8.7G).\n- `GSE305146_seur_integr_labelled_exc_lin_PCW10_20.rda.gz` (listing: 7.6G).\n\nNeither was downloaded or inspected here. Their names alone do not prove that\nthey contain the later `peaks_by_cluster` assay, compatible counts, or an external\ncommon feature space. A published object can close within-study measurement\ngaps without closing cross-study compatibility or training-only peak-selection\nrequirements. Downloading either whole object is outside this check\'s 20 MB cap.\n\nThe [NeMO ATAC directory](https://data.nemoarchive.org/other/grant/r21_delatorre/delatorre/multimodal/sncell/10xMultiome_ATACseq/human/processed/counts/)\nlists the DSdevctx ATAC MEX and fragment archive, but no separate small fixed-\nregion count object. The inspected raw GEO peak-list mismatch remains as\ndocumented in [the existing audit](PAIRED_MULTIOME_AUDIT.md).\n\n**Next actionable input:** establish the exact assays in a provider-documented\ncommon-count checkpoint and compare measured region/count semantics; otherwise\nscope an exact fragment recount on frozen regions with measured disk/network\nbudgets. A different GPU cannot resolve missing measurements or provenance.\n\n## 2026-09-09 update: public transport recovered, scientific gates unchanged\n\nThe previous transport blocker is cleared for the tested endpoints. Verified HTTPS\nnow returns the ATAC manifest, both collection APIs and both official Open release\nbags. The live [ATAC API](https://assets.nemoarchive.org/api/collection/nemo:col-ad8t52b)\nand [RNA API](https://assets.nemoarchive.org/api/collection/nemo:col-mbgxwtz) explicitly\nreport `access: open`. NeMO\'s [ATAC Open bag](https://data.nemoarchive.org/publication_release/Vuong_delaTorre_Human_snMultiome-2026-05-04/Analysis_bag_3_Vuong_delaTorre_Human_snMultiome_Analysis_ATAC_Open.tgz)\nand [RNA Open bag](https://data.nemoarchive.org/publication_release/Vuong_delaTorre_Human_snMultiome-2026-05-04/Analysis_bag_4_Vuong_delaTorre_Human_snMultiome_Analysis_RNA_Open.tgz)\nlist the DSdevctx processed counts. These support public analysis-release access;\nthe manifest\'s conflicting `Access=embargo` field is retained, not hidden. Neither\nAPI supplies a license, and no redistribution grant is inferred.\n\nThe Open bags declare ATAC counts of 1,540,753,269 bytes, RNA counts of 717,659,049\nbytes, and metadata of 4,618,240 bytes. The existing local metadata\'s MD5 exactly\nmatches the release declaration, `b48ba7434b20db66d89447899ef72b15`. This ties the\nprior audit to the released metadata; it does not certify retained-cell QC.\n\nThe complete bag inventories add no separate QC rule, retained-barcode artifact,\nor cross-cohort fixed-region count matrix. Barcodes inside the large count archives\nwere not inspected. The checked Lattke source tree is unchanged at\n`227f51b4e63c6a7d9c73be44f06ab21ac11e45ba`. Thus `ANNOTATION_COUNT_MATCH_QC_UNVERIFIED`\nand the common-measured-region requirement remain open; do not start confirmatory\nneural training or treat unmatched peaks as zeros.\n\n[Recheck record](../reports/generated/paired_public_recheck_2026-09-09/verification.md)\ncontains exact URLs, response hashes and limits. Checked at 2026-09-10 02:21:26 UTC\n(September 9 PDT). Direct metadata traffic was approximately 69 KB and bounded\nbelow 1.26 MB; browser/search response sizes were not exposed. No count/fragment\npayload, author outreach, controlled access or paid compute was used. Raw response\nsnapshots were not retained; their measured hashes and findings are recorded.\n\n## Verification record (2026-09-08 historical checks)\n\n- Public [RNA child collection](https://assets.nemoarchive.org/collection/nemo:col-mbgxwtz)\n  and [ATAC child collection](https://assets.nemoarchive.org/collection/nemo:col-ad8t52b)\n  are described as open processed releases. Parent collection remains restricted\n  because it also contains controlled data. No redistribution permission inferred.\n- Repeated direct public NeMO manifest download failed TLS with curl exit 35;\n  cached directory/collection listings remained readable. No matrix downloaded.\n- Follow-up transport checks reached TCP but closed during TLS negotiation, before\n  a server certificate or HTTP response. TLS 1.2 with both curl and Python/OpenSSL\n  failed; plain HTTP returned an empty reply. The documented public asset API also\n  failed TLS. This establishes a transport blocker here, not its server-side cause;\n  no certificate checks were disabled and no supported working mirror was found.\n  The official [RNA manifest](https://data.nemoarchive.org/other/grant/r21_delatorre/delatorre/multimodal/sncell/10xMultiome_RNAseq/human/processed/col-mbgxwtz-Vuong_delaTorre_Human_snMultiome_Analysis_RNA-Open-manifest.tsv)\n  is under `/processed/`; the [ATAC manifest](https://data.nemoarchive.org/other/grant/r21_delatorre/delatorre/multimodal/sncell/10xMultiome_ATACseq/human/processed/counts/col-ad8t52b-Vuong_delaTorre_Human_snMultiome_Analysis_ATAC-Open-manifest.tsv)\n  is under `/processed/counts/`. Do not conflate their directory layouts.\n- PMC article HTML returned a browser challenge. The documented\n  [NCBI E-utilities route](https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pmc&id=13225313)\n  returned the actual manuscript XML with DOI `10.1126/science.aea1259`.\n  Saved locally as `data/multiome/Vuong_PMC13225313_efetch_20260908.xml`:\n  467,378 bytes; SHA-256\n  `7e58d9f00c3e7132145aa4903661f33be3f265b9c9b6712389c1a1eac1af005d`.\n- Supplementary Table S1 request returned a browser-challenge HTML page, not an\n  Excel workbook; it was not parsed as data or used as evidence. Response retained\n  as `data/multiome/Vuong_Table_S1_20260908.failed.html`. No challenge bypassed.\n- Metadata source remains\n  `data/multiome/VuongWeber_DSdevctx_metadata.tar`, SHA-256\n  `72cf7284663aeaee76ed3bab16ce0b414faf82847a883e946e7c2afab21b1526`.\n  Selected CSV member:\n  `VuongWeber_2025_DSdevctx_metadata_20260128/VuongWeber_2025_DSdevctx_metadata_20260128.csv.gz`.\n- Sorted RNA non-`Unk` original cell IDs, joined by newline without a trailing\n  newline, SHA-256:\n  `0de060f67ab7d355e90766ab81168929260b3ce43d663b27d53f82c47f5ff9d6`.\n  Corresponding excluded-ID hash:\n  `30a274ee9f17278c1f4f020b1cf3bf2889a3b89c0733746f066ebd5f96749d16`.\n\nReproduce the new metadata checks without expression or ATAC matrices:\n\n```bash\n.venv-p22/bin/python - <<\'PY\'\nimport hashlib\nimport tarfile\nfrom pathlib import Path\nimport pandas as pd\nfrom p22.data.census import sha256_file\n\npath = Path(\'data/multiome/VuongWeber_DSdevctx_metadata.tar\')\nassert sha256_file(path) == \'72cf7284663aeaee76ed3bab16ce0b414faf82847a883e946e7c2afab21b1526\'\nmember = (\'VuongWeber_2025_DSdevctx_metadata_20260128/\'\n          \'VuongWeber_2025_DSdevctx_metadata_20260128.csv.gz\')\nwith tarfile.open(path) as archive:\n    frame = pd.read_csv(archive.extractfile(member), compression=\'gzip\')\nassert frame[\'Unnamed: 0\'].is_unique\nunknown = frame[\'class\'].eq(\'Unk\')\nassert unknown.equals(frame[\'cluster.ids\'].eq(\'Unk\'))\nretained = frame.loc[~unknown]\nassert (len(frame), int(unknown.sum()), len(retained)) == (117532, 3731, 113801)\nassert retained.groupby(\'condition\').donor.nunique().to_dict() == {\'Ctrl\': 13, \'Ts21\': 13}\nassert (frame.groupby(\'donor\')[[\'condition\', \'source\', \'gw\', \'sex\', \'ancestry\']].nunique() == 1).all().all()\nassert frame.drop_duplicates(\'donor\').source.value_counts().to_dict() == {\'UCLA\': 18, \'NIH_NBB\': 8}\nassert int((retained.nCount_ATAC <= 100).sum()) == 1751\nassert int((retained[\'percent.mt\'] >= 5).sum()) == 6\ndigest = hashlib.sha256(\'\\n\'.join(sorted(retained[\'Unnamed: 0\'])).encode()).hexdigest()\nassert digest == \'0de060f67ab7d355e90766ab81168929260b3ce43d663b27d53f82c47f5ff9d6\'\nprint(\'Candidate count reconciliation and provider/QC checks passed; scientific acceptance not implied.\')\nPY\n```\n\nResearch skill used to restrict findings to primary sources and preserve exact\nreproducible evidence; caveman skill used for concise task updates.\n', 'docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md': "# External RNA replication: real-data results\n\nCompleted locally on 2026-09-08 (run timestamps are 2026-09-09 UTC).\nFresh Colab verification and final archive comparison completed 2026-09-09.\n\n## Outcome\n\nAll four frozen comparisons and the headline are **inconclusive**. The code and\nreal experiment and all five original RNA plan tasks are complete. Fresh Colab\nexecutions, archived hashes and local result comparison passed. This is an\nactual summary-effect comparison, not a synthetic demonstration. It neither\nestablishes directional replication nor proves that reproducible effects are absent.\n\nEach comparison used nine DS and eight control discovery donors, with exact PCW\nand sex adjustment. The external publication has five DS and five control donors;\nonly its published differential-expression summaries were used.\n\n| Discovery → external | Cells | Shared genes | Spearman rho (95% donor-bootstrap CI) | Top-100 direction agreement | Upper permutation p | Outcome |\n|---|---:|---:|---|---:|---:|---|\n| RG → oRG | 16,855 | 9,051 | -0.0079 (-0.2516, 0.3187) | 58% | 0.01499 | Inconclusive |\n| RG → vRG | 16,855 | 8,572 | -0.0305 (-0.2425, 0.2644) | 53% | 0.18282 | Inconclusive |\n| RG_prol ∪ IPC_prol → CP | 8,417 | 9,213 | 0.0967 (-0.1806, 0.3349) | 59% | 0.04096 | Inconclusive |\n| IPC → IP | 10,621 | 7,857 | 0.0097 (-0.1920, 0.2642) | 53% | 0.10390 | Inconclusive |\n\nEach row completed 1,000 class-stratified donor bootstraps and 1,000 whole-vector\npermutations, with zero failed bootstraps. The oRG agreement p-value alone does\nnot satisfy the frozen support rule: its rho interval crosses zero. RG cells\nappear in two comparisons and must not be counted twice as independent evidence.\nAll lower-tail p-values, agreement intervals and audit fields are in the CSV.\n\nThe [frozen plan](EXTERNAL_RNA_REPLICATION_PLAN.md) defines gene exclusions,\nexpression floors, top-gene reranking, resampling and descriptive headline rules.\nThe headline rule carries no guaranteed 2.5% family-wise error calibration.\n\n## Final evidence package\n\nThese local generated artifacts are gitignored, not committed or uploaded:\n\n- [Executed canonical notebook](../reports/generated/notebooks/P22_external_rna_20260908_final.executed.ipynb).\n- [Four-row results and audit columns](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/runs/real_analysis_20260728/validation.csv) (filter `analysis=external_rna_direction_replication`).\n- [Four-panel concordance figure](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/runs/real_analysis_20260728/figures/external_rna_concordance.png).\n- [Notebook summary](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/notebook_summary.json), [gate board](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/gate_board.json), [manifest](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/runs/real_analysis_20260728/manifest.json), [run report](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/runs/real_analysis_20260728/SUMMARY.md).\n- [Full test output](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/runs/real_analysis_20260728/verification/test-all.log), [lint/format output](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/runs/real_analysis_20260728/verification/lint.log), [Colab status](../reports/generated/external_rna_20260908_final/run_20260909T044905Z/runs/real_analysis_20260728/verification/colab-status.md).\n\nFinal execution used code commit `75485ca319401806a8d7bdd20fd2e91d804528dd`.\nSubsequent commits record the corrected test wording and this evidence; they do\nnot change the scientific code or results. The nested `real_analysis_20260728`\ndirectory is a legacy run label, not the actual execution date.\n\n| Input or artifact | Bytes where applicable | SHA-256 |\n|---|---:|---|\n| Primary `f16c25da-15bd-46a4-9a3f-17093f27a2f1.h5ad` | 1,569,658,860 | `08d6eff265db6e6a2e1c4a259153588f3dba3c51f5f754736dc63c28795fcdbb` |\n| External `41467_2025_63752_MOESM4_ESM.xlsx` | 19,576,790 | `ea0e5a0d96e122ce39ace8be29b65039c3f7775cc96699873a8acc4ee74ccf13` |\n| Canonical notebook | — | `819604b370d8784303764ab0e2dbfe4e3a975c88b9a6df750321b7e0c92caf9f` |\n| Final executed notebook | — | `78f74008869606798623718f1eaa826f24ce2727807629446c95fcaa426369b7` |\n\nWorkbook schema, duplicate handling and chromosome-21 orientation passed all four\nsheets. Discovery began with 248,998 cells/30 donors; existing RNA QC retained\n248,960 cells/30 donors. The prespecified age/population filters yield the 17\ndiscovery donors and population sizes above. Source:\n[published study](https://www.nature.com/articles/s41467-025-63752-0) and\n[Supplementary Data 2](https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-63752-0/MediaObjects/41467_2025_63752_MOESM4_ESM.xlsx).\n\n## Verification and resources\n\n- 608 tests passed; 19 pre-existing warnings. Ruff checks and formatting passed\n  for 101 Python files. Statistical tests include independently reconstructed\n  OLS/bootstrap outputs, rank failures, expression floors and hash/schema guards.\n- Three complete local executions had zero notebook errors. Every RNA result\n  row matched exactly across runs (maximum absolute numeric difference: zero).\n  This is reproducibility checking, not three independent biological experiments.\n- Final executed cell sources exactly match the 23-cell canonical notebook.\n  Regeneration is idempotent; hand-maintained cells 2, 3 and 5 are preserved.\n- Final notebook execution: 155.885 seconds. RNA stage: 39.702 seconds.\n  Process peak RSS: 4.4118 GB, including preceding notebook stages; this is not\n  an isolated RNA memory-allocation measurement.\n- Earlier runs remain under `external_rna_20260908/run_20260909T044111Z` and\n  `external_rna_20260908_repeat/run_20260909T044602Z`. Their RNA numbers match;\n  their general G8 wording is superseded by the final package linked above.\n- G8 remains `INCONCLUSIVE`. Its evidence now records the verified processed RNA\n  result while stating that no external raw matrix was ingested. Existing G1\n  catalog pairing uncertainty and legacy G6 `PASS` do not establish paired\n  multimodal success. Historical approval policy remains unchanged; the\n  [user-reported attestation](../plan/real_data_attestation_2026-09-08.json) is explicit.\n\nActual final invocation, from the repository root, with the existing local\nPython 3.11 kernel installed by `make notebook-kernel`:\n\n```bash\nP22_NOTEBOOK_MODE=real_analysis \\\nP22_RUN_REAL_DATA=1 \\\nP22_RUN_MODEL=1 \\\nP22_PROFESSOR_APPROVED=1 \\\nP22_OUTPUT_ROOT=/Users/anuragdani/Github/niw-eb1a/P22/reports/generated/external_rna_20260908_final \\\nJUPYTER_PATH=/Users/anuragdani/Github/niw-eb1a/P22/reports/generated/jupyter \\\nMPLCONFIGDIR=/private/tmp/p22-matplotlib \\\nOPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 VECLIB_MAXIMUM_THREADS=1 \\\n.venv-p22/bin/python -m jupyter nbconvert \\\n  --to notebook --execute P22_down_syndrome_all_in_one.ipynb \\\n  --ExecutePreprocessor.kernel_name=python3 \\\n  --ExecutePreprocessor.timeout=1800 \\\n  --ExecutePreprocessor.startup_timeout=300 \\\n  --output-dir reports/generated/notebooks \\\n  --output P22_external_rna_20260908_final.executed.ipynb\n```\n\nThe manifest's existing `exact_command` field is a reconstructed default template\nand omits this run's output/kernel/thread settings. The audited invocation above\nis the exact command record. Generated manifests have not been silently edited.\nFor another run, choose a new output name to preserve the final executed copy.\n\n## Fresh Colab verification (complete 2026-09-09)\n\nThe user explicitly confirmed Professor Fang's approval and separately approved\nupload of the canonical notebook and a free CPU run. This confirmation is saved\nin the scoped attestation; do not request the same approval again. No independent\napproval date was invented, and scientific acceptance gates remain unchanged.\n\n[Saved Colab notebook](https://colab.research.google.com/drive/1aQgaq5cq7A5tOxHMYeCKaPGAG2Htxc3c).\nOnly the canonical notebook was uploaded from this machine. The VM fetched public\ninputs; no Drive mount, paid compute, GPU, or controlled-access request was used.\n\n- Native Python **3.13.15** completed the scientific notebook and displayed four\n  inconclusive RNA rows. G0 stayed `INCONCLUSIVE` because the project requires\n  Python 3.11. Native RNA time was 183.037 seconds; process peak RSS was 2.8256 GB.\n- A separate **Python 3.11.16** environment on the same free Colab CPU VM executed\n  the unchanged canonical notebook through nbconvert. This was a Colab-hosted\n  subprocess, not the native Colab kernel; its `in_colab=False` is not rewritten.\n  G0 passed. Final successful invocation took 664.508 seconds; RNA stage took\n  163.391 seconds, with 2.0403 GB process peak RSS. This timing includes notebook\n  execution, not initial environment setup; RSS includes preceding stages.\n- Independent checks of the downloaded archive confirmed all 23 cell sources\n  exactly match the canonical notebook, every code cell executed, and zero errors\n  in the executed 3.11 notebook. Both Colab runs' four RNA rows, scientific fields\n  and cohort counts match the frozen local run exactly: maximum numeric difference\n  **zero**. G8 and all RNA outcomes remain inconclusive.\n- Initial 3.11 startup failed because inherited Colab configuration requested\n  `google.colab._kernel.Kernel`, then a Google extension unavailable in the isolated\n  environment. Private IPython/Jupyter configuration directories and explicit\n  standard IPython kernel/empty extension settings fixed startup. Both failures,\n  the passing smoke check and successful final execution remain in the VM logs.\n  No scientific source or native Colab security setting was changed.\n- The native download is saved locally as\n  [P22_colab_native.executed.ipynb](../reports/generated/colab_verification_2026-09-09/P22_colab_native.executed.ipynb).\n  It predates final 3.11 completion; it is not the final combined archive.\n  Its SHA-256 is `accbe892e359a4ebb2c948ee4158868df4a62f75cd21c1c7daea8b68e6cb7e61`.\n  [Independent local inspection](../reports/generated/colab_verification_2026-09-09/native-only-check.json)\n  confirms the exact 23-cell source sequence, all 12 canonical code cells executed,\n  zero scientific-cell errors, and an exact match for all four RNA rows and outer\n  scientific fields (maximum numeric difference zero; timing/RSS/figure excluded).\n  The earlier auxiliary 3.11 startup error remains visible, separate from those\n  scientific cells. The final snapshot and pinned-runtime archive are now retained\n  separately below; this earlier native check is historical evidence.\n- The separate [focused test log](../reports/generated/colab_verification_2026-09-09/focused-tests.log)\n  records 15 passing tests. Full-suite/lint records above remain valid; scientific\n  code did not change during this verification.\n\nSuccessful 3.11 invocation on the VM:\n\n```bash\n/content/p22_py311/bin/jupyter-nbconvert --to notebook --execute \\\n  /content/P22_down_syndrome_all_in_one.ipynb \\\n  --ExecutePreprocessor.kernel_name=p22-py311 \\\n  --ExecutePreprocessor.timeout=1800 \\\n  --ExecutePreprocessor.startup_timeout=300 \\\n  --output-dir /content/reports/generated/notebooks/py311 \\\n  --output P22_colab_py311.executed.ipynb\n```\n\nIts environment enabled `real_analysis`, real data/model and the scoped professor\nattestation; data root was `/content/p22_data`, output root was\n`/content/reports/generated/p22_colab_py311_verification`, BLAS/OpenMP/MKL threads\nwere one, and `CUDA_VISIBLE_DEVICES` was empty. Full arguments, package versions,\ntimings and environment are in the generated `colab_verification_record` folder.\n\nThe downloaded [verification archive](../reports/generated/colab_verification_2026-09-09/P22_colab_verification_20260909.zip) contains\n40 generated/approved-source files, **1,264,546 bytes**, SHA-256\n`5b26965f218510aa96c123cb07c829c44ba35967a8439229b2b5587f8ae5dfac`.\nThe export includes both run folders, the executed 3.11 notebook, canonical source,\nlogs and per-file hashes, but no raw datasets. Local archive size/SHA and all 39\nlisted file hashes pass; the fortieth file is the inventory itself, covered by the\nZIP hash. Extraction rejected unsafe paths, symlinks and duplicate names and did\nnot overwrite existing files. [Integrity record](../reports/generated/colab_verification_2026-09-09/archive-integrity.json).\n\n- [Python 3.11 executed notebook](../reports/generated/colab_verification_2026-09-09/archive/reports/generated/notebooks/py311/P22_colab_py311.executed.ipynb): SHA-256 `14002e327e4d7b6cfcc1b457f874c857ae8aed5c88ca70a38e8916d065c02c8a`.\n- [Final saved Colab snapshot](../reports/generated/colab_verification_2026-09-09/P22_colab_final.downloaded.ipynb): 2,679,354 bytes; SHA-256 `5141f209670dc0674e56bfff27462fe2e203792256df914ef68ff2da96907e92`.\n- [Exact execution/environment record](../reports/generated/colab_verification_2026-09-09/archive/reports/generated/colab_verification_record/execution_record.json), [final execution log](../reports/generated/colab_verification_2026-09-09/archive/reports/generated/colab_verification_record/execute_canonical_python311_isolated.log), [file hash inventory](../reports/generated/colab_verification_2026-09-09/archive/reports/generated/colab_verification_record/verification_checks.json).\n- [Passing comparison](../reports/generated/colab_verification_2026-09-09/comparison.json), [command output](../reports/generated/colab_verification_2026-09-09/comparison.log), [completed Colab status](../reports/generated/colab_verification_2026-09-09/colab-status.md).\n\nReproduce the comparison against retained evidence:\n\n```bash\n.venv-p22/bin/python reports/generated/colab_verification_2026-09-09/compare_colab.py \\\n  reports/generated/colab_verification_2026-09-09/archive\n```\n\nThe generated comparator checks original notebook sources, execution/errors,\nmanifest source hashes, all four RNA records and cohort fields, G0 and G8 against\nthe frozen local run. It reports all differences, with numeric tolerance 1e-9;\nonly outer timing, memory and figure paths are excluded. The report passes with\nno failures or pending checks. Task 5 is complete; the native Python-version\ndifference, isolated-kernel startup failures and resource differences are retained.\n\nRetrieval was resolved without another analysis run: the normal Google Drive\ndownload contained the saved archive link's embedded ZIP. The notebook was copied\nthrough Finder, decoded locally and checked against the previously recorded ZIP\nhash. No OS permissions were expanded. Historical pending status files and earlier\nstartup errors are preserved; this completed record supersedes them.\n\n## Claim limits\n\nThis compares aggregate rank/sign directions against published cell-level MAST\nsummaries with a donor covariate. It is not raw external reanalysis, per-gene\nreplication, external predictive validation, ATAC validation, or a neural routing\nmechanism result. Batch remains unmodelled; primary PCW 19 is absent; nominal\npopulation labels are not proven equivalent. `PCW18_DS_16545` has sub-quality\ntissue, and `PCW17_CON_14310` has relatively thin population support. Natural-log\ndiscovery effects and external log2 effects are compared only by rank/sign.\nRemoving chromosome 21 does not establish dosage-independent causation.\n\n## Completion ledger and remaining action\n\n| Plan component | State | Remaining requirement |\n|---|---|---|\n| Original RNA tasks 1–4: implementation and local execution | Complete | None locally; retain inconclusive result |\n| Original RNA task 5: verification | Complete | Fresh Colab evidence archived; source/hash/result comparison passes |\n| Paired M1–M4: accepted real inputs | Partial/blocked | Common measured ATAC counts and accepted release/QC semantics |\n| Paired M5: real scientific protocol | Pending | Freeze preprocessing, covariates, biological controls and sensitivities on accepted inputs |\n| Paired M6–M8: real neural comparison | Pending | Real internal fits and locked external scoring after upstream gates pass |\n\nNo original RNA task remains. Colab sign-in, authorized upload, free CPU execution,\narchive retrieval and independent local comparison are complete. This closes the\nRNA plan without changing its inconclusive scientific conclusion.\n\nFor paired work, the [source and transport audit](PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md)\nrecords the exact NeMO annotation count match and unresolved QC semantics. Its\nSeptember 9 recheck confirms public transport/release access now works; that clears\nthe earlier TLS blocker, not QC or cross-cohort feature acceptance.\nThe [paired checklist](../tasks/todo.md) retains all real-data requirements.\nAccept a documented common-count release or separately approve a bounded fragment\nrecount with adequate storage; do not zero-fill unmeasured regions or silently\nstart the roughly 43 GiB compressed fragment collection. Bigger GPU alone cannot\nclose these input gates. No paid service, author outreach or controlled access\nwas initiated.\n", 'docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md': '# RNA donor influence: results, 2026-09-10\n\nPOST_HOC_EXPLORATORY. This completes the bounded F4–F5 RNA diagnostic, not the\nfull P22 plan. Original RNA results remain inconclusive; G8 is unchanged.\nNo paired predictive evaluation or real neural training ran.\n\n## Result\n\nAll four saved baselines reproduced exactly, including donor order and saved\ngene/donor/cell counts. Every donor was omitted once per comparison: 68/68\nomissions available, each with design rank 4 and 12 residual degrees of freedom.\n\n`PCW18_DS_16545` had the largest absolute same-support influence in each comparison.\nThe same donors contribute to every comparison; RG comparisons also share cells.\nThese are not four independent observations about that donor.\n\n| Comparison | Baseline rho | Same-support delta omitting this donor | Omission rho | Share of absolute influence |\n|---|---:|---:|---:|---:|\n| RG → oRG | -0.007896 | +0.191714 | 0.183818 | 20.0% |\n| RG → vRG | -0.030550 | +0.165192 | 0.134642 | 19.5% |\n| cycling progenitors → CP | 0.096712 | +0.169408 | 0.266120 | 20.4% |\n| IPC → IP | 0.009666 | +0.118950 | 0.128616 | 16.0% |\n\nThese shares describe concentration; they are not significance thresholds.\nFor this donor, the support delta was zero in all four comparisons. Across all\n68 omissions, the largest absolute support delta was 0.000756456. At most five\nbaseline genes were lost and at most one new gene became eligible per omission.\nNew genes were counted but excluded from the diagnostic endpoint.\n\nDecision: keep all donors and preserve the original conclusion. Do not select\ndonor exclusions to improve agreement. External-donor uncertainty remains\nunresolved. Advance the separately planned paired-input feasibility tasks before\nproposing real training; this diagnostic supplies neither common measured ATAC\ncounts nor accepted external QC.\n\n## Execution and verification\n\n- Implementation commit: `a0049f2960f796871ed9ed7b4971077200d48ae7`.\n- Frozen config commit: `5be576c`; canonical config SHA256:\n  `d51a69cc997fe3a80615ea76bdf9d9504a5334c05fe09eb8fe77a623d266cf58`.\n- One real CPU run: 45.942 seconds; peak sampled RSS 4.141571 GiB.\n  Limit: 1,800 seconds, stop at 5 GiB under a 6 GiB budget. Sampled monitoring\n  is not a kernel hard ceiling. Both run and resource reports say `COMPLETED`.\n- Final focused checks: 20 passed. Final full suite: 628 passed in 64.63 seconds;\n  19 existing synthetic classification warnings. Lint/format: 108 files pass.\n  Final fast suite: 596 passed, 32 deselected in 59.58 seconds.\n- Fixtures check planted influence, stable data, absent class, rank failure,\n  insufficient/constant/nonfinite effects, normalization, expression-floor support\n  changes, decomposition, repeated-output determinism, refusal before omissions,\n  all-unavailable reporting, no overwrite, and owned-child timeout/memory races.\n- Original H5AD, workbook, reference CSV and four reused source helpers have\n  identical before/after hashes. All 12 original dirty-file/notebook fingerprints\n  matched after execution; original checkout status remains unchanged.\n- Main verified resource completion, 68 unique rows, fit rank/df, delta arithmetic,\n  implementation/artifact hashes and plot readability. Independent artifact\n  review passed: exact baseline/Gi correlations, donor/class accounting, gene-set\n  fingerprints and ranks; delta arithmetic error at most 1.39e-17. Full review\n  is recorded in the delivered package. No second real refit was run; omission\n  refit correctness rests on reviewed code and synthetic checks, not independent\n  raw-count re-estimation during artifact review.\n\nThe local model supplied only a small test draft, which needed supervisor\ncorrection. Core implementation and scientific decisions were supervisor-led\nafter explicit user approval; independent reviewers checked code and results.\nThe rejected local-only pilot report remains historical, not an accepted protocol.\n\n## Artifacts and reproduction\n\nSee [the one-command guide](RNA_DONOR_INFLUENCE.md). Results are in\n`reports/generated/rna_donor_influence_20260910_supervised/`: full donor table,\nbaseline genes, plot, summary, run/resource records and empty successful-run logs.\nThe original checkout receives an identical durable artifact copy plus a reviewed\nGit bundle. No original evidence is overwritten; no push or merge is performed.\nThe bundle retains the isolated commits, not ignored source data or the shared\nenvironment. Preserve the exact reference CSV and config paths when reproducing.\n'}
EXPECTED_SHA256 = {'notebooks/P22_paired_workflow.ipynb': '623ac92645c10cc47e5d96b147d989c2b08eb4d6ec0507ce2c9fc5b480b932d4', 'pyproject.toml': '997c84cca3f1bcf0711bdd6b48f17d109c00a61db5aea96b2a8d9fb2205ff8ee', 'scripts/launcher_head_capture.py': 'fd721b8d8522003422223dd38f2bffbb388033ecb4f0c6af369198602fde61e6', 'scripts/capture_development_head.py': '335d35c51b27d48704eaad62a695d74e6e91f456e3148b15abe8d3aff846029c', 'scripts/validate_development_head.py': 'c761489202fdbfaba5153f7a508aefa19028cf87f681bd473382c6c4c713ac9d', 'tests/test_development_head.py': 'a5dbeedadd380b3ccb5802eab5d4449935c2300af71a1a5bc53bafacdbe32c77', 'tests/test_development_head_capture.py': 'd011e88988c6d70ae313211aae2fda1f88449a03a365937a9f82cbce752baca2', 'tests/test_launcher_head_capture.py': 'b3ab13cf0c6ceb9ad1b1ffdfd2718237385dc8901e1fcf3c9210b30c09f12fcc', 'configs/development_object_source_contract.json': '9a13d8beafa5e00d81f8f985649a6a466c6b4fe3ebf008206f840c32316113f2', 'docs/PAIRED_MULTIOME_REMAINING_EVIDENCE_2026-09-08.md': '66b1d8158f5b6c4260085cddb736db2d59de9df610e5dcac5ccfc2c4f63b18b9', 'docs/EXTERNAL_RNA_REPLICATION_RESULTS_2026-09-08.md': '81bb3c313749f4b2751e6259a0c52544d7b10762baddc80597abd36521b96354', 'docs/RNA_DONOR_INFLUENCE_RESULTS_2026-09-10.md': 'a8892ffd454898bcdc5594c79e09d55d07d70398be5ce3f8f9b58cde563edd36'}
ROOT = Path(tempfile.mkdtemp(prefix="p22-paired-colab-")).resolve()
for name, content in PAYLOAD.items():
    relative = Path(name)
    assert not relative.is_absolute() and ".." not in relative.parts
    raw = content.encode("utf-8")
    assert len(raw) <= 262144
    assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256[name]
    destination = ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("xb") as handle:
        handle.write(raw)
(ROOT / "reports/generated").mkdir(parents=True)
print("Embedded files verified:", len(PAYLOAD), "Root:", ROOT)
print("Only offline code, tests, and saved reports; no raw datasets.")


In [ ]:
#@title Prepare Python 3.11 (CPU only)
# Keep the native Colab kernel unchanged. Use Python 3.11 for P22.
import shutil
if sys.version_info[:2] == (3, 11):
    P22_PYTHON = sys.executable  # Local verification uses the existing P22 environment.
else:
    assert Path("/content").is_dir(), "Use Python 3.11 locally, or run in Colab."
    uv = shutil.which("uv")
    if uv is None:
        tools_dir = ROOT / "uv-tools"
        subprocess.run([sys.executable, "-m", "pip", "install", "--only-binary=:all:",
                        "--target", str(tools_dir), "uv==0.12.13"], check=True, timeout=180)
        uv = str(tools_dir / "bin/uv")
    subprocess.run([uv, "venv", "--python", "3.11", str(ROOT / "venv")],
                   check=True, timeout=240)
    P22_PYTHON = str(ROOT / "venv/bin/python")
    subprocess.run([uv, "pip", "install", "--python", P22_PYTHON, "--only-binary=:all:",
                    "pytest==8.4.2", "nbformat==5.10.4", "nbclient==0.11.0",
                    "ipykernel==7.3.0"], check=True, timeout=240)
version_check = "import sys; assert sys.version_info[:2] == (3,11); print(sys.version)"
subprocess.run([P22_PYTHON, "-c", version_check],
               check=True, timeout=15)
print("Environment ready. No ML stack or datasets installed.")


In [ ]:
#@title Execute the unchanged notebook and verify outputs
# Private configuration avoids inheriting Colab's incompatible kernel extensions.
environment = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
environment.pop("PYTHONPATH", None)
for key in ("IPYTHONDIR", "JUPYTER_CONFIG_DIR"):
    directory = ROOT / key.lower()
    directory.mkdir(exist_ok=True)
    environment[key] = str(directory)
run = subprocess.run([P22_PYTHON, "-c", 'import json, os, sys\nfrom datetime import UTC, datetime\nfrom importlib.metadata import version\nfrom pathlib import Path\nimport nbformat\nfrom nbclient import NotebookClient\n\nassert sys.version_info[:2] == (3, 11)\nroot = Path.cwd()\nsource = nbformat.read(root / "notebooks/P22_paired_workflow.ipynb", as_version=4)\nbefore = [(c.id, c.source) for c in source.cells]\nclient = NotebookClient(source, kernel_name="python3", timeout=120, allow_errors=False)\nmanager = client.create_kernel_manager()\nmanager.kernel_spec.argv = [sys.executable, "-m", "ipykernel_launcher", "-f", "{connection_file}"]\ntry:\n    client.execute(cwd=str(root), env=dict(os.environ))\nfinally:\n    nbformat.write(source, root / "P22_paired_workflow.executed.ipynb")\nassert before == [(c.id, c.source) for c in source.cells]\ncode = [c for c in source.cells if c.cell_type == "code"]\nassert [c.execution_count for c in code] == list(range(1, 10))\nassert not [o for c in code for o in c.outputs if o.output_type == "error"]\ntests = "".join(o.get("text", "") for o in code[1].outputs)\nassert "181 passed" in tests\ndecision = json.loads("".join(o.get("text", "") for o in code[-1].outputs))\nassert decision["scientific_outcome"] == "INCONCLUSIVE"\nassert decision["training_performed"] is False and decision["live_allowed"] is False\nassert decision["live_requests_attempted"] == 0\nrecord = {"completed_at": datetime.now(UTC).isoformat(), "python": sys.version,\n          "source_cells_unchanged": True, "executed_code_cells": len(code),\n          "packages": {p: version(p) for p in ["pytest", "nbformat", "nbclient", "ipykernel"]},\n          "offline_test_output": tests, "decision": decision}\n(root / "verification.json").write_text(json.dumps(record, indent=2))\nprint(json.dumps(record, indent=2))\n'], cwd=ROOT,
                     env=environment, capture_output=True, text=True, timeout=240)
print(run.stdout)
if run.stderr:
    print(run.stderr)
run.check_returncode()


In [ ]:
#@title Show results and package evidence
from IPython.display import Markdown, display
executed = json.loads((ROOT / "P22_paired_workflow.executed.ipynb").read_text())
for cell in executed["cells"]:
    if cell["cell_type"] == "markdown":
        display(Markdown("".join(cell["source"])))
    else:
        for output in cell.get("outputs", []):
            if output["output_type"] == "stream":
                print("".join(output["text"]))
            elif "data" in output:
                display({k: "".join(v) if isinstance(v, list) else v
                         for k, v in output["data"].items()}, raw=True)
import zipfile
ARCHIVE = ROOT / "P22_paired_colab_verification.zip"
with zipfile.ZipFile(ARCHIVE, "x", zipfile.ZIP_DEFLATED) as archive:
    for name in (*PAYLOAD, "P22_paired_workflow.executed.ipynb", "verification.json"):
        archive.write(ROOT / name, name)
print("VERIFIED: 9 unchanged code cells, 181 offline tests; scientific result still INCONCLUSIVE.")
print("Evidence archive:", ARCHIVE)


In [ ]:
from google.colab import files
files.download(str(ARCHIVE))